# 15 · scikit-learn — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch15_sklearn/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch15_sklearn/03_notebook.ipynb`, créée par `python tools/start_chapter.py 15`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 15.Q1 à 15.Q12, 15.R1 à 15.R3, 15.1 à 15.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 15.9 | Anatomie d'un estimateur : Ridge sur un sinus bruité | 📦 | ★ | 15 |
|  | 15.10 | predict_proba : prédire ce que valent les probabilités de trois classifieurs | 🔮 | ★ | 10 |
|  | 15.11 | Datasets synthétiques, fetch_* et train_test_split stratifié | 📦 | ★ | 15 |
|  | 15.12 | Faire tourner le code de 2018 avec scikit-learn 1.6 | 🐛 | ★★ | 25 |
| B | 15.13 | Clustering avec scikit-learn : KMeans (le coude du livre), DBSCAN et clustering hiérarchique sur deux lunes et sept blobs | 📦 | ★★ | 25 |
|  | 15.14 | Transformateurs : test hors de [0, 1] et inverse_transform | 📦 | ★★ | 20 |
|  | 15.15 | PCA de 3D vers 2D et blanchiment | 📦 | ★★ | 20 |
|  | 15.16 | Cinq spirales : quel ensemble les démêle ? | 🔬 | ★★ | 30 |
| C | 15.17 | cross_validate, KFold, StratifiedKFold et RidgeClassifierCV | 📦 | ★★ | 25 |
|  | 15.18 | GridSearchCV : best_params_, cv_results_ et score d'entraînement | 📦 | ★★ | 25 |
|  | 15.19 | RandomizedSearchCV avec des distributions | 📦 | ★★ | 25 |
|  | 15.20 | Grille ou hasard : qui gagne avec 16 essais ? | 🔮 | ★★ | 20 |
|  | 15.21 | Frontières de décision : DecisionBoundaryDisplay et surfaces 3D | 📦 | ★★ | 20 |
|  | 15.22 | Pipeline polynomial + ridge : degré et alpha ensemble | 📦 | ★★★ | 35 |
|  | 15.23 | Lire cv_results_ : carte de chaleur et écarts-types | 📈 | ★★ | 20 |
|  | 15.24 | Refaire l'expérience de fuite du ch. 8 (sélection de features sur du bruit) avec un Pipeline | 🔬 | ★★ | 25 |
| D | 15.25 | Régler le seuil de décision d'un classifieur | 🔬 | ★★★ | 35 |
|  | 15.26 | Classes déséquilibrées : class_weight, rééchantillonnage et la bonne métrique (recall, PR-AUC) | 📦 | ★★★ | 35 |
|  | 15.27 | La même validation croisée pour six familles de modèles scikit-learn : score moyen, écart-type et temps | 📦 | ★★★ | 40 |
|  | 15.28 | ColumnTransformer sur Penguins brut | 📦 | ★★★ | 45 |
|  | 15.29 | Quand scikit-learn refuse un estimateur mylearn | 🐛 | ★★★ | 35 |
|  | 15.30 | Sauvegarder, recharger et versionner un pipeline | 🛠️ | ★★ | 20 |
|  | 15.31 | Défi : prédire le sexe des manchots depuis le fichier brut | 🏆 | ★★★★ | 100 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="15")  # your own library

## Objectifs et rappel express

- Utiliser l'interface commune de scikit-learn : hyperparamètres au constructeur, `fit` qui renvoie l'objet, attributs appris suffixés par `_`, `predict`, `decision_function`, `predict_proba`, `transform` et `inverse_transform`.
- Évaluer et régler un modèle : `cross_val_score`, `cross_validate`, `KFold`, `StratifiedKFold`, `GridSearchCV`, `RandomizedSearchCV`, et lire `cv_results_`.
- Enchaîner sans fuite : `Pipeline`, `make_pipeline`, `ColumnTransformer`, noms `étape__paramètre`.
- Moderniser du code ancien, choisir un seuil de décision, traiter des classes déséquilibrées, sauvegarder un pipeline avec ses versions.

**Rappel express.** $\hat p = \sigma(s) = 1/(1 + e^{-s})$ à deux classes, le softmax à $K$ classes ; nombre d'entraînements d'une grille : $\big(\prod_h n_h\big) \times k + 1$ ; probabilité qu'au moins un de $n_{\text{iter}}$ essais au hasard tombe dans le meilleur $q$ : $1 - (1 - q)^{n_{\text{iter}}}$ ; nombre de features polynomiales : $\binom{p + d}{d}$ ; min-max : $x' = (x - m)/(M - m)$. En scikit-learn : `sklearn.model_selection` (validation croisée, recherches, découpage), `sklearn.pipeline`, `sklearn.compose`, `sklearn.preprocessing`, `sklearn.inspection`, `sklearn.calibration` ; `joblib` pour la sauvegarde.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 15.1 à 15.4 et 🧮 15.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un nom : une chaîne (`"standardscaler"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la réflexion (🗣️ 15.6, ⚖️ 15.7, 📄 15.8) et l'entretien se corrigent avec `05_solutions.md`.

**15.Q1 — Coder soi-même ou utiliser une bibliothèque ?**

In [ ]:
# 15.Q1: the values you found on paper
answer_15_Q1a = ...  # a) True or False
answer_15_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_15_Q1c = ...  # c) True or False
answer_15_Q1d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_15_Q1a, answer_15_Q1b, answer_15_Q1c, answer_15_Q1d]):
    wb.check(f"15.Q1{letter}", answer)

**15.Q2 — Importer proprement et lire une dépréciation**

In [ ]:
# 15.Q2: the values you found on paper
answer_15_Q2a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q2b = ...  # b) the letter of your choice, e.g. "E"
answer_15_Q2c = ...  # c) True or False
answer_15_Q2d = ...  # d) the letter of your choice, e.g. "E"
answer_15_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_Q2a, answer_15_Q2b, answer_15_Q2c, answer_15_Q2d, answer_15_Q2e]):
    wb.check(f"15.Q2{letter}", answer)

**15.Q3 — Hyperparamètres au constructeur, attributs appris avec « _ »**

In [ ]:
# 15.Q3: the values you found on paper
answer_15_Q3a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q3b = ...  # b) True or False
answer_15_Q3c = ...  # c) True or False
answer_15_Q3d = ...  # d) the letter of your choice, e.g. "E"
answer_15_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_Q3a, answer_15_Q3b, answer_15_Q3c, answer_15_Q3d, answer_15_Q3e]):
    wb.check(f"15.Q3{letter}", answer)

**15.Q4 — Ce que renvoient fit et predict**

In [ ]:
# 15.Q4: the values you found on paper
answer_15_Q4a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q4b = ...  # b) the letter of your choice, e.g. "E"
answer_15_Q4c = ...  # c) the letter of your choice, e.g. "E"
answer_15_Q4d = ...  # d) True or False
answer_15_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_Q4a, answer_15_Q4b, answer_15_Q4c, answer_15_Q4d, answer_15_Q4e]):
    wb.check(f"15.Q4{letter}", answer)

**15.Q5 — decision_function ou predict_proba ?**

In [ ]:
# 15.Q5: the values you found on paper
answer_15_Q5a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q5b = ...  # b) the letter of your choice, e.g. "E"
answer_15_Q5c = ...  # c) the letter of your choice, e.g. "E"
answer_15_Q5d = ...  # d) a number
answer_15_Q5e = ...  # e) True or False
answer_15_Q5f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_15_Q5a, answer_15_Q5b, answer_15_Q5c, answer_15_Q5d, answer_15_Q5e, answer_15_Q5f]):
    wb.check(f"15.Q5{letter}", answer)

**15.Q6 — KMeans : n_clusters, labels_ et choix de k**

In [ ]:
# 15.Q6: the values you found on paper
answer_15_Q6a = ...  # a) a whole number
answer_15_Q6b = ...  # b) True or False
answer_15_Q6c = ...  # c) the letter of your choice, e.g. "E"
answer_15_Q6d = ...  # d) True or False
answer_15_Q6e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_15_Q6a, answer_15_Q6b, answer_15_Q6c, answer_15_Q6d, answer_15_Q6e]):
    wb.check(f"15.Q6{letter}", answer)

**15.Q7 — fit, transform, fit_transform, inverse_transform**

In [ ]:
# 15.Q7: the values you found on paper
answer_15_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q7b = ...  # b) a number
answer_15_Q7c = ...  # c) a number
answer_15_Q7d = ...  # d) a number
answer_15_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_Q7a, answer_15_Q7b, answer_15_Q7c, answer_15_Q7d, answer_15_Q7e]):
    wb.check(f"15.Q7{letter}", answer)

**15.Q8 — Ce que la PCA fait vraiment aux features**

In [ ]:
# 15.Q8: the values you found on paper
answer_15_Q8a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q8b = ...  # b) True or False
answer_15_Q8c = ...  # c) True or False
answer_15_Q8d = ...  # d) True or False
answer_15_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_Q8a, answer_15_Q8b, answer_15_Q8c, answer_15_Q8d, answer_15_Q8e]):
    wb.check(f"15.Q8{letter}", answer)

**15.Q9 — Un ensemble est un estimateur comme les autres**

In [ ]:
# 15.Q9: the values you found on paper
answer_15_Q9a = ...  # a) True or False
answer_15_Q9b = ...  # b) the letter of your choice, e.g. "E"
answer_15_Q9c = ...  # c) the letter of your choice, e.g. "E"
answer_15_Q9d = ...  # d) True or False
answer_15_Q9e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_15_Q9a, answer_15_Q9b, answer_15_Q9c, answer_15_Q9d, answer_15_Q9e]):
    wb.check(f"15.Q9{letter}", answer)

**15.Q10 — Grille ou recherche aléatoire ?**

In [ ]:
# 15.Q10: the values you found on paper
answer_15_Q10a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q10b = ...  # b) True or False
answer_15_Q10c = ...  # c) a whole number
answer_15_Q10d = ...  # d) a whole number
answer_15_Q10e = ...  # e) True or False
answer_15_Q10f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_15_Q10a, answer_15_Q10b, answer_15_Q10c, answer_15_Q10d, answer_15_Q10e, answer_15_Q10f]):
    wb.check(f"15.Q10{letter}", answer)

**15.Q11 — Pipeline : step__param et absence de fuite**

In [ ]:
# 15.Q11: the values you found on paper
answer_15_Q11a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q11b = ...  # b) True or False
answer_15_Q11c = ...  # c) the letter of your choice, e.g. "E"
answer_15_Q11d = ...  # d) the letter of your choice, e.g. "E"
answer_15_Q11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_Q11a, answer_15_Q11b, answer_15_Q11c, answer_15_Q11d, answer_15_Q11e]):
    wb.check(f"15.Q11{letter}", answer)

**15.Q12 — Datasets, train_test_split et documentation**

In [ ]:
# 15.Q12: the values you found on paper
answer_15_Q12a = ...  # a) a number
answer_15_Q12b = ...  # b) True or False
answer_15_Q12c = ...  # c) the letter of your choice, e.g. "E"
answer_15_Q12d = ...  # d) True or False
answer_15_Q12e = ...  # e) a whole number
answer_15_Q12f = ...  # f) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdef", [answer_15_Q12a, answer_15_Q12b, answer_15_Q12c, answer_15_Q12d, answer_15_Q12e, answer_15_Q12f]):
    wb.check(f"15.Q12{letter}", answer)

**15.R1 — Forêt aléatoire ou gradient boosting ?**

In [ ]:
# 15.R1: the values you found on paper
answer_15_R1a = ...  # a) the letter of your choice, e.g. "E"
answer_15_R1b = ...  # b) True or False
answer_15_R1c = ...  # c) the letter of your choice, e.g. "E"
answer_15_R1d = ...  # d) the letter of your choice, e.g. "E"
answer_15_R1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_R1a, answer_15_R1b, answer_15_R1c, answer_15_R1d, answer_15_R1e]):
    wb.check(f"15.R1{letter}", answer)

**15.R2 — Où ajuster le scaler dans une validation croisée ?**

In [ ]:
# 15.R2: the values you found on paper
answer_15_R2a = ...  # a) True or False
answer_15_R2b = ...  # b) the letter of your choice, e.g. "E"
answer_15_R2c = ...  # c) a number
answer_15_R2d = ...  # d) True or False
answer_15_R2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_15_R2a, answer_15_R2b, answer_15_R2c, answer_15_R2d, answer_15_R2e]):
    wb.check(f"15.R2{letter}", answer)

**15.R3 — Combien de modèles pour une validation croisée à 5 folds ?**

In [ ]:
# 15.R3: the values you found on paper
answer_15_R3a = ...  # a) a whole number
answer_15_R3b = ...  # b) a whole number
answer_15_R3c = ...  # c) a whole number
answer_15_R3d = ...  # d) a whole number
answer_15_R3e = ...  # e) a whole number
answer_15_R3f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_15_R3a, answer_15_R3b, answer_15_R3c, answer_15_R3d, answer_15_R3e, answer_15_R3f]):
    wb.check(f"15.R3{letter}", answer)

**Ex 15.1 — Compter les modèles entraînés par une recherche en grille**

In [ ]:
# Ex 15.1: the values you found on paper
answer_15_1a = ...  # a) a whole number
answer_15_1b = ...  # b) a whole number
answer_15_1c = ...  # c) a whole number
answer_15_1d = ...  # d) a whole number
answer_15_1e = ...  # e) a whole number
answer_15_1f = ...  # f) a whole number
answer_15_1g = ...  # g) a whole number
answer_15_1h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_15_1a, answer_15_1b, answer_15_1c, answer_15_1d, answer_15_1e, answer_15_1f, answer_15_1g, answer_15_1h]):
    wb.check(f"15.1{letter}", answer)

**Ex 15.2 — De decision_function à predict_proba**

In [ ]:
# Ex 15.2: the values you found on paper
answer_15_2a = ...  # a) a number with 4 decimals
answer_15_2b = ...  # b) a number with 4 decimals
answer_15_2c = ...  # c) a number
answer_15_2d = ...  # d) a list of 5 classes, e.g. [1, 1, 1, 1, 1]
answer_15_2e = ...  # e) a number with 4 decimals
answer_15_2f = ...  # f) a number with 4 decimals
answer_15_2g = ...  # g) a whole number
answer_15_2h = ...  # h) a number with 4 decimals
answer_15_2i = ...  # i) True or False
answer_15_2j = ...  # j) True or False

for letter, answer in zip("abcdefghij", [answer_15_2a, answer_15_2b, answer_15_2c, answer_15_2d, answer_15_2e, answer_15_2f, answer_15_2g, answer_15_2h, answer_15_2i, answer_15_2j]):
    wb.check(f"15.2{letter}", answer)

**Ex 15.3 — Ramener une droite ajustée dans l'espace d'origine**

In [ ]:
# Ex 15.3: the values you found on paper
answer_15_3a = ...  # a) a number
answer_15_3b = ...  # b) a number
answer_15_3c = ...  # c) a number
answer_15_3d = ...  # d) a number
answer_15_3e = ...  # e) a number
answer_15_3f = ...  # f) a number
answer_15_3g = ...  # g) a number
answer_15_3h = ...  # h) a number
answer_15_3i = ...  # i) a number
answer_15_3j = ...  # j) True or False

for letter, answer in zip("abcdefghij", [answer_15_3a, answer_15_3b, answer_15_3c, answer_15_3d, answer_15_3e, answer_15_3f, answer_15_3g, answer_15_3h, answer_15_3i, answer_15_3j]):
    wb.check(f"15.3{letter}", answer)

**Ex 15.4 — Combien de features polynomiales ?**

In [ ]:
# Ex 15.4: the values you found on paper
answer_15_4a = ...  # a) a whole number
answer_15_4b = ...  # b) a whole number
answer_15_4c = ...  # c) a whole number
answer_15_4d = ...  # d) a whole number
answer_15_4e = ...  # e) a whole number
answer_15_4f = ...  # f) a whole number
answer_15_4g = ...  # g) a whole number
answer_15_4h = ...  # h) a number with 2 decimals
answer_15_4i = ...  # i) a whole number

for letter, answer in zip("abcdefghi", [answer_15_4a, answer_15_4b, answer_15_4c, answer_15_4d, answer_15_4e, answer_15_4f, answer_15_4g, answer_15_4h, answer_15_4i]):
    wb.check(f"15.4{letter}", answer)

**Ex 15.5 — Budget d'une recherche d'hyperparamètres**

In [ ]:
# Ex 15.5: the values you found on paper
answer_15_5a = ...  # a) a whole number
answer_15_5b = ...  # b) a number with 1 decimal
answer_15_5c = ...  # c) a number with 1 decimal
answer_15_5d = ...  # d) a number with 1 decimal
answer_15_5e = ...  # e) a number with 4 decimals
answer_15_5f = ...  # f) a whole number
answer_15_5g = ...  # g) a number
answer_15_5h = ...  # h) a number with 1 decimal

for letter, answer in zip("abcdefgh", [answer_15_5a, answer_15_5b, answer_15_5c, answer_15_5d, answer_15_5e, answer_15_5f, answer_15_5g, answer_15_5h]):
    wb.check(f"15.5{letter}", answer)

## Partie A · Un estimateur et ses sorties

L'interface d'un estimateur sur l'exemple du livre (15.9), ses trois sorties à prédire (15.10), les datasets et le découpage (15.11), puis le code de 2018 à faire tourner avec la version 1.6 (15.12). La cellule suivante importe ce dont les parties A à D ont besoin, définit leurs outils et découpe les 333 manchots complets (233 pour l'entraînement, 100 pour le test), comme aux ch. 13 et 14. Ne recopie jamais un nombre à la main : chaque réponse est calculée par ton code, et pas arrondie.

In [ ]:
# Tools for the notebook exercises (parts A to D), and the penguins
import contextlib
import json
import platform
import tempfile
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from scipy import stats
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone, is_classifier
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.datasets import load_wine, make_blobs, make_circles, make_classification, make_moons
from sklearn.decomposition import PCA
from sklearn.ensemble import AdaBoostClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, RidgeClassifier, RidgeClassifierCV
from sklearn.metrics import (accuracy_score, adjusted_rand_score, average_precision_score, balanced_accuracy_score,
                             confusion_matrix, mean_squared_error, precision_score, recall_score, root_mean_squared_error,
                             silhouette_score)
from sklearn.model_selection import (GridSearchCV, KFold, RandomizedSearchCV, StratifiedKFold, cross_val_score,
                                     cross_validate, train_test_split)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from wb.errors import MylearnMissingError


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def filled(*values):
    """True when none of the values is still `...` (or None): the answer has been written."""
    return all(value is not ... and value is not None for value in values)


def fr(x, digits=3):
    """A number written the French way, with a decimal comma (for the messages in French)."""
    return f"{x:.{digits}f}".replace(".", ",")


def returned(ex_id, name, value):
    """False, with a ❌ message, when the function `name` returned None (a forgotten return); True otherwise."""
    if value is None:
        print(f"❌ Ex {ex_id} : {name} renvoie None : as-tu oublié le return ?")
        return False
    return True


def print_answer(ex_id, value, **_):
    """Solutions notebook: show the value that the exercise notebook checks with wb.check."""
    try:
        shown = (np.round(np.asarray(value, dtype=float), 4).tolist() if isinstance(value, (np.ndarray, list, tuple))
                 else value)
    except (TypeError, ValueError):     # a list of names
        shown = list(value)
    print(f"{ex_id}:", shown)


@contextlib.contextmanager
def checked(ex_id):
    """Like wb.attempt (⏳ while the code is not written yet), but any other error raised by YOUR code becomes one ❌
    line instead of stopping the cell."""
    with wb.attempt(ex_id):
        try:
            yield
        except (NotImplementedError, ImportError, MylearnMissingError):
            raise
        except Exception as exc:                              # a bug in the learner's code: report it, go on
            if isinstance(exc, TypeError) and "NoneType" in str(exc):
                raise                                         # wb.attempt explains the forgotten return
            hint = " ; une fonction ou une méthode oublie-t-elle son return ?" if "NoneType" in str(exc) else ""
            print(f"❌ Ex {ex_id} : ton code lève {type(exc).__name__} : {str(exc)[:300]}{hint}")


def as_float(value):
    """A number given as a float, an int, a NumPy scalar or a one-element array, as a Python float; None otherwise."""
    try:
        array = np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        return None
    return float(array.reshape(-1)[0]) if array.size == 1 else None


PENGUINS = wb.datasets.load_penguins(dropna=True).reset_index(drop=True)   # the 333 complete penguins, sorted by species
MEASURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
P_TRAIN, P_TEST = train_test_split(PENGUINS, test_size=100, random_state=1314, stratify=PENGUINS["species"])
print(len(P_TRAIN), "training penguins,", len(P_TEST), "test penguins")

### Ex 15.9 — Anatomie d'un estimateur : Ridge sur un sinus bruité 📦 ★ ⏱️ 15 min
**Objectif :** créer, entraîner et interroger un estimateur en respectant l'interface de scikit-learn.  
**Prérequis :** ch. 9 (Ridge) · 0A (tableaux NumPy, `reshape`) · fiche §15.4 · **Fil rouge :** synthétique · **Parcours :** R, M, C

L'exemple du livre (§15.4, figures 15.2 et 15.3) : cent points tirés autour d'une courbe sinus (`x_9`, `y_9`, abscisses dans le désordre), et une droite de `Ridge` ajustée dessus. **Contrat** (ce que vérifie la cellule) :
a) `X_9` : `x_9` sous forme de matrice `X`, de forme `(100, 1)` ;
`model_9` : un `Ridge(alpha=1.0)` entraîné sur `X_9` et `y_9` ;
b) `slope_9` et c) `intercept_9` : sa pente et son ordonnée à l'origine, deux nombres (`float`) lus dans ses attributs appris ;
d) `ends_9` : la liste des deux prédictions de `model_9` aux extrémités `x_9.min()` et `x_9.max()` (un seul appel à `predict`) ;
e) `r2_9` : le $R^2$ de `model_9` sur ses propres données d'entraînement (la méthode `score`) ;
f) `strong_slope_9` : la pente d'une **copie** de `model_9` (`sklearn.base.clone`) passée à `alpha=100.0` par `set_params`, puis entraînée sur les mêmes données (`model_9` ne doit pas changer) ;
g) `half_slope_9` : la pente d'un **nouveau** `Ridge(alpha=1.0)` entraîné seulement sur les points où `x_9 < 2.5`.

La vérification contrôle a) à g) et que `model_9` n'a pas changé, puis trace les points, la droite et celle de f) (passée par le point moyen, pour comparer les pentes).

Dans tes notes : pourquoi `predict` refuse-t-il `x_9.min()` seul ? Pourquoi la pente diminue-t-elle en valeur absolue avec `alpha=100.0` ? Que serait devenu `model_9` si tu avais appelé `set_params` sur lui plutôt que sur une copie ?

In [ ]:
rng_9 = np.random.default_rng(1509)
x_9 = rng_9.uniform(0, 5, 100)                       # 100 abscissas, in no particular order
y_9 = np.sin(x_9) + rng_9.normal(0, 0.15, 100)       # noisy points around a sine

In [ ]:
X_9 = ...             # a) x_9 as a 2-D array of shape (100, 1): one row per example, one column per feature
model_9 = ...         #    Ridge(alpha=1.0), fitted on X_9 and y_9
slope_9 = ...         # b) its slope, a float (read it in a learned attribute)
intercept_9 = ...     # c) its intercept, a float
ends_9 = ...          # d) list of 2 floats: the predictions of model_9 at x_9.min() and at x_9.max()
r2_9 = ...            # e) the R² of model_9 on its own training data
strong_slope_9 = ...  # f) the slope of a copy of model_9 (sklearn.base.clone) set to alpha=100.0 with set_params,
                      #    then fitted on the same data
half_slope_9 = ...    # g) the slope of a NEW Ridge(alpha=1.0) fitted only on the points with x_9 < 2.5

In [ ]:
if filled(X_9) and np.ndim(X_9) != 2:
    print(f"❌ Ex 15.9a : X_9 doit être à deux dimensions, de forme (100, 1) (une ligne par exemple, une colonne par feature) ; reçu la forme {np.shape(X_9)}.")
else:
    wb.check("15.9a", list(np.shape(X_9)) if filled(X_9) else X_9)
wb.check("15.9b", slope_9)
wb.check("15.9c", intercept_9)
wb.check("15.9d", ends_9)
wb.check("15.9e", r2_9)
wb.check("15.9f", strong_slope_9)
wb.check("15.9g", half_slope_9)
with wb.attempt("15.9"):
    if filled(model_9) and hasattr(model_9, "coef_") and filled(slope_9) and as_float(slope_9) is not None:
        verdict("15.9", getattr(model_9, "alpha", None) == 1.0 and np.isclose(np.ravel(model_9.coef_)[0], as_float(slope_9)),
                "model_9 garde alpha=1.0 et la pente slope_9",
                "model_9 a changé (alpha ou pente) : f) se fait sur une copie, clone(model_9), pas sur model_9 lui-même")
    if not filled(slope_9, intercept_9):
        raise NotImplementedError("slope_9, intercept_9")
    if as_float(slope_9) is None or as_float(intercept_9) is None:
        print("❌ Ex 15.9 : slope_9 et intercept_9 doivent être des nombres (coef_ est un tableau : prends son élément).")
    else:
        line_x_9 = np.array([0.0, 5.0])
        fig, ax = plt.subplots(figsize=(6, 3.6))
        ax.scatter(x_9, y_9, s=12, label="noisy sine")
        ax.plot(line_x_9, as_float(slope_9) * line_x_9 + as_float(intercept_9), color="black", label="Ridge, alpha = 1")
        if filled(strong_slope_9) and as_float(strong_slope_9) is not None:
            ax.plot(line_x_9, as_float(strong_slope_9) * (line_x_9 - x_9.mean()) + y_9.mean(), ls="--",
                    label="alpha = 100 (through the mean point)")
        ax.set(xlabel="x", ylabel="y")
        ax.legend(fontsize=8)
        plt.show()

### Ex 15.10 — predict_proba : prédire ce que valent les probabilités de trois classifieurs 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir ce que valent vraiment les probabilités de trois classifieurs, au-delà de leurs formes.  
**Prérequis :** ✏️ 15.2 · Ex 15.9 · ch. 3 (calibration) · ch. 13 (SVM, arbres, Naïve Bayes) · fiche §15.4.4 · **Fil rouge :** Penguins · **Parcours :** R, C

Les formes des sorties sont dans la fiche ; ce qui suit porte sur leurs **valeurs**. Trois modèles déjà entraînés sur les 233 manchots d'entraînement :
- `svc_10` : un `SVC(probability=True, random_state=0)` (après un `StandardScaler`) sur les deux mesures du bec ;
- `tree_10` : un `DecisionTreeClassifier(random_state=0)` **sans limite de profondeur**, sur les 4 mesures ;
- `nb_10` et `logistic_10` : un `GaussianNB()` et une régression logistique (après un `StandardScaler`), sur les 4 mesures.

Prédis, **avant** d'exécuter quoi que ce soit :
a) `disagree_10` : sur une grille de 40 000 points qui couvre les mesures du bec des manchots (`GRID_10`), la part des points où `svc_10.predict` ne donne **pas** la classe de plus grande probabilité de `svc_10.predict_proba` : `"A"` aucun ; `"B"` quelques-uns, moins de 5 % ; `"C"` de 5 à 20 % ; `"D"` plus de 20 % ;
b) `pure_10` : la part des 100 manchots de test dont la plus grande probabilité de `tree_10.predict_proba` vaut **exactement** 1 : `"A"` moins de la moitié ; `"B"` de la moitié à 90 % ; `"C"` plus de 90 %, mais pas tous ; `"D"` tous ;
c) `nb_bolder_10` : la moyenne, sur les manchots de test, de la plus grande probabilité donnée par Naïve Bayes est-elle **plus grande** que celle de la régression logistique ? `True` ou `False`.

Puis exécute l'expérience : elle mesure les trois quantités, affiche les accuracies et marque sur la carte des classes du SVC les points de désaccord.

In [ ]:
BILL_10 = ["bill_length_mm", "bill_depth_mm"]
svc_10 = make_pipeline(StandardScaler(), SVC(probability=True, random_state=0)).fit(P_TRAIN[BILL_10], P_TRAIN["species"])
tree_10 = DecisionTreeClassifier(random_state=0).fit(P_TRAIN[MEASURES], P_TRAIN["species"])
nb_10 = GaussianNB().fit(P_TRAIN[MEASURES], P_TRAIN["species"])
logistic_10 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(P_TRAIN[MEASURES], P_TRAIN["species"])
# a grid of 200 x 200 = 40,000 points that covers the bill measures of all the penguins (2 mm and 1 mm beyond)
g1_10 = np.linspace(PENGUINS[BILL_10[0]].min() - 2, PENGUINS[BILL_10[0]].max() + 2, 200)
g2_10 = np.linspace(PENGUINS[BILL_10[1]].min() - 1, PENGUINS[BILL_10[1]].max() + 1, 200)
GRID_10 = pd.DataFrame(np.array(np.meshgrid(g1_10, g2_10)).reshape(2, -1).T, columns=BILL_10)

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
disagree_10 = ...   # a) "A", "B", "C" or "D"
pure_10 = ...       # b) "A", "B", "C" or "D"
nb_bolder_10 = ...  # c) True or False

In [ ]:
if any(answer is ... or answer is None for answer in [disagree_10, pure_10, nb_bolder_10]):
    print("⏳ Ex 15.10 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    labels_10 = svc_10.predict(GRID_10)
    argmax_10 = svc_10.classes_[svc_10.predict_proba(GRID_10).argmax(axis=1)]
    share_10 = float(np.mean(labels_10 != argmax_10))
    print(f"a) SVC(probability=True): predict and the argmax of predict_proba disagree on {share_10:.2%} of the grid")
    top_tree_10 = tree_10.predict_proba(P_TEST[MEASURES]).max(axis=1)
    print(f"b) unlimited tree: {np.mean(top_tree_10 == 1.0):.0%} of the test penguins get a largest probability of exactly 1 "
          f"(accuracy {tree_10.score(P_TEST[MEASURES], P_TEST['species']):.2f})")
    top_nb_10 = nb_10.predict_proba(P_TEST[MEASURES]).max(axis=1).mean()
    top_lr_10 = logistic_10.predict_proba(P_TEST[MEASURES]).max(axis=1).mean()
    print(f"c) mean largest probability: Naive Bayes {top_nb_10:.3f}, logistic regression {top_lr_10:.3f} · accuracies: "
          f"{nb_10.score(P_TEST[MEASURES], P_TEST['species']):.2f} and {logistic_10.score(P_TEST[MEASURES], P_TEST['species']):.2f}")
    fig, ax = plt.subplots(figsize=(6, 4.5))
    codes_10 = pd.Categorical(labels_10, categories=svc_10.classes_).codes
    ax.contourf(g1_10, g2_10, codes_10.reshape(200, 200), levels=[-0.5, 0.5, 1.5, 2.5], alpha=0.25, colors=wb.plot.class_colors(3))
    ax.scatter(*GRID_10[labels_10 != argmax_10].to_numpy().T, s=2, color="black", label="predict ≠ argmax of predict_proba")
    for k, species in enumerate(svc_10.classes_):
        ax.scatter(*P_TRAIN[P_TRAIN["species"] == species][BILL_10].to_numpy().T, s=12, color=wb.plot.class_colors(3)[k], label=species)
    ax.set(xlabel=BILL_10[0], ylabel=BILL_10[1])
    ax.legend(fontsize=8)
    plt.show()
    wb.check("15.10a", disagree_10)
    wb.check("15.10b", pure_10)
    wb.check("15.10c", nb_bolder_10)

Dans tes notes : compare avec tes prédictions. D'où viennent les probabilités d'un `SVC`, d'un arbre, d'un Naïve Bayes ? Lesquelles prendrais-tu pour décider d'un seuil, et que faudrait-il vérifier avant (ch. 3) ?

### Ex 15.11 — Datasets synthétiques, fetch_* et train_test_split stratifié 📦 ★ ⏱️ 15 min
**Objectif :** charger les trois familles de datasets de scikit-learn, et voir ce que font le mélange et la stratification.  
**Prérequis :** Ex 15.9 · ch. 8 (découpage, stratification) · fiche §15.10 et §15.11 · **Fil rouge :** synthétique · **Parcours :** C

Trois familles de datasets : embarqués (`load_*`), téléchargés puis gardés en cache (`fetch_*`), générés (`make_*`). Le workbook charge California par `wb.datasets.load_california`, qui appelle `fetch_california_housing` et garde une copie de secours. **Contrat** :
a) `wine_11` : la liste `[nombre d'exemples, nombre de features, nombre de classes]` de `load_wine` ;
b) `first_feature_11` : le nom de sa première feature (une chaîne, dans le `Bunch`) ;
c) `california_11` : la liste `[nombre d'exemples, nombre de features]` de `X`, avec `X, y = wb.datasets.load_california(return_X_y=True)` ;
d) `no_shuffle_11` : `X_circ_11`, `y_circ_11` sont deux cercles générés **dans l'ordre** (`make_circles(..., shuffle=False)`, comme les points du livre au §15.11) ; la part de la classe 1 dans la partie **test** de `train_test_split(X_circ_11, y_circ_11, test_size=0.25, shuffle=False)` ;
e) `shuffled_11` : la même part, avec le mélange par défaut et `random_state=0` ;
f) `stratified_11` : la même part, avec `random_state=0` et `stratify=y_circ_11` ;
g) `rare_plain_11` : `X_rare_11`, `y_rare_11` comptent environ 10 % d'exemples de la classe 1 ; le **nombre** d'exemples de la classe 1 dans le test de `train_test_split(X_rare_11, y_rare_11, test_size=0.25, random_state=0)` ;
h) `rare_strat_11` : le même nombre avec `stratify=y_rare_11`.

La vérification contrôle a) à h), puis montre le test des cercles sans et avec mélange.

Dans tes notes : pourquoi le test de d) ne ressemble-t-il pas à l'entraînement ? Avec 20 exemples de la classe 1, combien en attendais-tu dans un quart des données ? Quand faut-il **ne pas** mélanger ?

In [ ]:
X_circ_11, y_circ_11 = make_circles(n_samples=400, noise=0.05, factor=0.5, shuffle=False, random_state=0)
X_rare_11, y_rare_11 = make_classification(n_samples=200, weights=[0.9], random_state=1511)   # about 10 % of class 1
print("circles:", X_circ_11.shape, "· rare:", int(y_rare_11.sum()), "examples of class 1 out of", len(y_rare_11))

In [ ]:
wine_11 = ...           # a) [n_samples, n_features, n_classes] of the embedded dataset load_wine (3 integers)
first_feature_11 = ...  # b) the name of the first feature of load_wine (a string)
california_11 = ...     # c) [n_samples, n_features] of California: X, y = wb.datasets.load_california(return_X_y=True)
no_shuffle_11 = ...     # d) share of class 1 in the TEST part of train_test_split(X_circ_11, y_circ_11, test_size=0.25,
                        #    shuffle=False)
shuffled_11 = ...       # e) the same share with the default shuffling and random_state=0
stratified_11 = ...     # f) the same share with random_state=0 and stratify=y_circ_11
rare_plain_11 = ...     # g) number of examples of class 1 in the test part of train_test_split(X_rare_11, y_rare_11,
                        #    test_size=0.25, random_state=0)
rare_strat_11 = ...     # h) the same number with stratify=y_rare_11

In [ ]:
wb.check("15.11a", wine_11)
wb.check("15.11b", first_feature_11)
wb.check("15.11c", california_11)
wb.check("15.11d", no_shuffle_11)
wb.check("15.11e", shuffled_11)
wb.check("15.11f", stratified_11)
wb.check("15.11g", rare_plain_11)
wb.check("15.11h", rare_strat_11)
if filled(no_shuffle_11, shuffled_11):          # the picture, once d) and e) are answered
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
    for ax, shuffle in zip(axes, (False, True)):
        X_a, X_b, y_a, y_b = train_test_split(X_circ_11, y_circ_11, test_size=0.25, shuffle=shuffle,
                                              random_state=0 if shuffle else None)
        ax.scatter(*X_a.T, s=8, c="lightgray", label="train")
        ax.scatter(*X_b.T, s=10, c=np.where(y_b == 1, "tab:orange", "tab:blue"), label="test")
        ax.set(title=f"shuffle={shuffle}", aspect="equal")
    axes[0].legend(fontsize=8)
    plt.show()

### Ex 15.12 — Faire tourner le code de 2018 avec scikit-learn 1.6 🐛 ★★ ⏱️ 25 min
**Objectif :** moderniser un vieux programme en lisant ses messages d'erreur et la documentation de la version installée.  
**Prérequis :** Ex 15.9 · ch. 9 (RMSE) · ch. 12 (one-hot) · fiche §15.3, §15.4.3 et §15.9.3 · **Fil rouge :** synthétique · **Parcours :** R, C

La fonction `study_12` recopie un notebook écrit en 2018 pour scikit-learn 0.19 : une droite et sa valeur à gauche, un encodage one-hot de couleurs, une recherche en grille sur un `RidgeClassifier`, la RMSE de la droite. Avec la version 1.6, elle plante, et même une fois réparée, elle ne redonne pas tout à fait les résultats de 2018 (`REPORT_12`, ce que le notebook affichait à l'époque).

Répare-la **sans changer ce qu'elle calcule** : elle doit tourner sans erreur et rendre les mêmes résultats que `REPORT_12`. Méthode : exécute la cellule de vérification, lis le message de l'erreur (son type, sa dernière ligne), cherche le paramètre ou l'attribut dans la documentation de **ta** version (`scikit-learn.org/1.6/`), corrige, recommence. La vérification numérote six points, dans l'ordre du programme ; tant qu'une erreur arrête le programme, les points suivants attendent (⏳) ; quand il tourne, elle compare chaque résultat au rapport (le point 6 porte sur la recherche : son meilleur score et ses scores d'entraînement). Quand les six points passent, elle relève a) le meilleur score de la recherche et b) la RMSE de la droite.

Dans tes notes : pour chaque point, le message reçu, la correction, et la version de scikit-learn qui a changé les choses (les notes de version le disent).

In [ ]:
rng_12 = np.random.default_rng(1512)
x_12 = rng_12.uniform(-2, 4, 60)                                   # 60 points, in no particular order
y_12 = 0.8 * x_12 + 1 + rng_12.normal(0, 0.5, 60)
colors_12 = rng_12.choice(np.array(["red", "green", "blue", "white"]), 60)
features_12, labels_12 = wb.synth.make_moons(n=240, noise=0.3, seed=1512)
REPORT_12 = {"y_left": -0.6287, "onehot_columns": 4, "best_alpha": 0.01, "best_score": 0.8417,
             "train_scores": [0.8458, 0.8458, 0.8438, 0.8375, 0.8042, 0.775],
             "rmse": 0.4741}   # what the 2018 notebook printed (scikit-learn 0.19), rounded to 4 decimals

In [ ]:
# The 2018 notebook: fix it (and keep what it computes)
def study_12(x, y, colors, features, labels):
    """A notebook written in 2018 for scikit-learn 0.19, turned into a function. It must run with scikit-learn 1.6 and
    give the same results as in 2018 (REPORT_12)."""
    # 1. a line through the points, and its value at the left end
    line = Ridge(alpha=1.0).fit(x.reshape(-1, 1), y)
    y_left = line.predict(x.min())
    # 2. the colours in one-hot
    encoder = OneHotEncoder(sparse=False)
    onehot = encoder.fit_transform(colors.reshape(-1, 1))
    # 3. a grid search on a RidgeClassifier
    search = GridSearchCV(RidgeClassifier(), {"alpha": [0.01, 0.1, 1, 10, 100, 1000]})
    search.fit(features, labels)
    best = search.best_parameters_
    train_scores = search.cv_results_["mean_train_score"]
    # 4. the root mean squared error of the line
    rmse = mean_squared_error(y, line.predict(x.reshape(-1, 1)), squared=False)
    return {"y_left": float(np.ravel(y_left)[0]), "onehot_columns": onehot.shape[1], "best_alpha": best["alpha"],
            "best_score": search.best_score_, "train_scores": list(train_scores), "rmse": float(rmse)}

In [ ]:
POINTS_12 = ["predict reçoit des données à deux dimensions", "OneHotEncoder accepte son argument de sortie dense",
             "le meilleur réglage se lit dans un attribut qui existe", "les scores d'entraînement de la recherche existent",
             "la RMSE se calcule sans argument retiré",
             "le meilleur score et les scores d'entraînement de la recherche sont ceux du rapport de 2018"]
CRASHES_12 = [(1, ValueError, "2D array"), (2, TypeError, "sparse"), (3, AttributeError, "best_parameters_"),
              (4, KeyError, "mean_train_score"), (5, TypeError, "squared")]


def crash_point_12(exc):
    """The number of the point that this exception reveals, or None."""
    for k, kind, text in CRASHES_12:
        if isinstance(exc, kind) and text in str(exc):
            return k
    return None


with checked("15.12"):
    try:
        result_12 = study_12(x_12, y_12, colors_12, features_12, labels_12)
        crash_12 = None
    except Exception as exc:                             # the 2018 code stops at its first incompatibility
        point_12 = crash_point_12(exc)
        if point_12 is None:
            raise                                        # another error: checked() reports it
        result_12, crash_12 = None, point_12
        for k in range(1, point_12):
            verdict("15.12", True, f"point {k} : {POINTS_12[k - 1]}.", "")
        print(f"❌ Ex 15.12 : point {point_12} ({POINTS_12[point_12 - 1]}) : le programme s'arrête sur "
              f"{type(exc).__name__} : {' '.join(str(exc).split())[:300]} Lis ce message, puis la documentation de ta version.")
        print(f"⏳ Ex 15.12 : points {point_12 + 1} à 6 : le programme doit d'abord aller plus loin.")
    if crash_12 is None and returned("15.12", "study_12", result_12):
        expected_12 = ["y_left", "onehot_columns", "best_alpha", "best_score", "train_scores", "rmse"]
        if not isinstance(result_12, dict) or sorted(result_12) != sorted(expected_12):
            print(f"❌ Ex 15.12 : study_12 doit rendre le dictionnaire de départ, avec les clés {expected_12}.")
        else:
            near_12 = lambda key: abs(float(result_12[key]) - REPORT_12[key]) < 6e-5
            try:
                train_12 = np.asarray(result_12["train_scores"], dtype=float).reshape(-1)
            except (TypeError, ValueError):
                train_12 = np.zeros(0)
            same_12 = lambda reference: train_12.shape == (6,) and np.allclose(train_12, reference, atol=6e-5, rtol=0)
            grid_12 = {"alpha": [0.01, 0.1, 1, 10, 100, 1000]}
            searches_12 = [GridSearchCV(RidgeClassifier(), grid_12, cv=folds, return_train_score=True).fit(
                features_12, labels_12).cv_results_ for folds in (3, 5)]          # the 0.19 and 1.6 defaults
            validation_12 = [results["mean_test_score"] for results in searches_12]   # not training scores
            checks_12 = [near_12("y_left"), result_12["onehot_columns"] == 4, result_12["best_alpha"] == 0.01,
                         train_12.shape == (6,) and bool(np.all((train_12 >= 0) & (train_12 <= 1)))
                         and not any(same_12(reference) for reference in validation_12), near_12("rmse"),
                         near_12("best_score") and same_12(REPORT_12["train_scores"])]
            hints_12 = ["la valeur à gauche n'est pas celle du rapport : la prédiction se fait-elle bien en x_12.min() ?",
                        "le rapport compte 4 colonnes (une par couleur)",
                        "le meilleur alpha n'est pas celui du rapport",
                        "il faut les 6 scores moyens d'entraînement de la recherche (un par valeur d'alpha), ceux que lisait le code de 2018, pas des scores de validation",
                        "la RMSE n'est pas celle du rapport : est-ce bien la racine de l'erreur quadratique moyenne ?",
                        f"le programme tourne, mais sa recherche ne redonne pas le rapport de 2018 (meilleur score "
                        f"{fr(float(result_12['best_score']), 4)} au lieu de {fr(REPORT_12['best_score'], 4)}, ou d'autres "
                        "scores d'entraînement) : un réglage par défaut a-t-il changé depuis 0.19 ? Relis la documentation de "
                        "GridSearchCV de ta version, et reprends exactement le découpage que faisait 0.19"]
            for k, (ok, hint) in enumerate(zip(checks_12, hints_12), start=1):
                verdict("15.12", ok, f"point {k} : {POINTS_12[k - 1]}.", f"point {k} : {hint}.")
            wb.check("15.12a", float(result_12["best_score"]), computed=True)
            wb.check("15.12b", float(result_12["rmse"]), computed=True)

## Partie B · Clustering, transformateurs, PCA et ensembles

Une visite des grandes familles d'outils, sur les exemples du livre : le clustering (15.13), les transformateurs et leur `inverse_transform` (15.14), la PCA et le blanchiment (15.15), AdaBoost sur cinq spirales (15.16). Les données sont synthétiques et fournies, avec leurs graines.

### Ex 15.13 — Clustering avec scikit-learn : KMeans (le coude du livre), DBSCAN et clustering hiérarchique sur deux lunes et sept blobs 📦 ★★ ⏱️ 25 min
**Objectif :** choisir un nombre de clusters avec l'inertie et la silhouette, et voir quand un clustering par densité ou hiérarchique fait mieux que k-means.  
**Prérequis :** ch. 7 (k-means, inertie, silhouette) · Ex 15.9 · fiche §15.5 · **Fil rouge :** synthétique · **Parcours :** C

Deux jeux : sept nuages gaussiens dont certains se chevauchent (`X_blobs_13`, l'exemple du livre aux figures 15.4 et 15.5), et deux demi-lunes peu bruitées (`X_moons_13`). Pour comparer un clustering aux vrais groupes, on utilise l'**indice de Rand ajusté** (`sklearn.metrics.adjusted_rand_score(vrais, trouvés)`) : il vaut 1 quand les deux partitions sont identiques (quels que soient les numéros des groupes), environ 0 pour un groupement au hasard.

**Contrat** :
a) `inertias_13` : pour chaque $k$ de `KS_13` (2 à 9), l'inertie d'un `KMeans(n_clusters=k, n_init=10, random_state=0)` entraîné sur les nuages ;
b) `silhouettes_13` : la silhouette moyenne (`silhouette_score`) de ces 8 clusterings ;
c) `best_k_13` : le $k$ de plus grande silhouette ;
d) `kmeans_ari_13` : l'indice de Rand ajusté d'un `KMeans(n_clusters=2, n_init=10, random_state=0)` sur les lunes ;
e) `dbscan_ari_13` : celui de `DBSCAN(eps=0.2)` sur les lunes ;
f) `small_eps_13` : la liste `[nombre de clusters, nombre de points de bruit]` de `DBSCAN(eps=0.1)` sur les lunes (le bruit a le label −1 et n'est pas un cluster) ;
g) `single_ari_13` : l'indice d'un `AgglomerativeClustering(n_clusters=2, linkage="single")` ;
h) `ward_ari_13` : celui d'un `AgglomerativeClustering(n_clusters=2)`, avec sa liaison par défaut (`"ward"`).

La vérification contrôle a) à h), puis trace les nuages, l'inertie et la silhouette, et quatre clusterings des lunes.

Dans tes notes : où vois-tu un coude ? La silhouette choisit-elle 7, et pourquoi ? Comment les quatre méthodes de d) à h) coupent-elles les lunes, et qu'est-ce qui, dans la façon dont chacune forme ses groupes, l'explique ? Que se passe-t-il quand `eps` est trop petit ?

In [ ]:
X_blobs_13, y_blobs_13 = make_blobs(n_samples=700, centers=7, cluster_std=1.0, center_box=(-10, 10),
                                    random_state=1513)          # seven Gaussian blobs, some of them overlapping
X_moons_13, y_moons_13 = make_moons(n_samples=300, noise=0.06, random_state=1513)
KS_13 = list(range(2, 10))

In [ ]:
inertias_13 = ...     # a) list: for each k of KS_13, the inertia_ of KMeans(n_clusters=k, n_init=10, random_state=0) fitted
                      #    on X_blobs_13
silhouettes_13 = ...  # b) list: the silhouette_score of these 8 clusterings (on X_blobs_13)
best_k_13 = ...       # c) the k of KS_13 with the largest silhouette
kmeans_ari_13 = ...   # d) adjusted_rand_score(y_moons_13, labels) for KMeans(n_clusters=2, n_init=10, random_state=0) on the moons
dbscan_ari_13 = ...   # e) the same for DBSCAN(eps=0.2) (min_samples left to its default)
small_eps_13 = ...    # f) list [number of clusters, number of noise points] for DBSCAN(eps=0.1) on the moons
                      #    (the noise, label -1, is not a cluster)
single_ari_13 = ...   # g) ARI of AgglomerativeClustering(n_clusters=2, linkage="single") on the moons
ward_ari_13 = ...     # h) ARI of AgglomerativeClustering(n_clusters=2) (its default linkage, "ward")

In [ ]:
wb.check("15.13a", inertias_13)
wb.check("15.13b", silhouettes_13)
wb.check("15.13c", best_k_13)
wb.check("15.13d", kmeans_ari_13)
wb.check("15.13e", dbscan_ari_13)
wb.check("15.13f", small_eps_13)
wb.check("15.13g", single_ari_13)
wb.check("15.13h", ward_ari_13)
with wb.attempt("15.13"):
    if not filled(inertias_13, silhouettes_13):
        raise NotImplementedError("inertias_13, silhouettes_13")
    try:
        shown_inertias_13 = np.asarray(inertias_13, dtype=float)
        shown_silhouettes_13 = np.asarray(silhouettes_13, dtype=float)
    except (TypeError, ValueError):
        shown_inertias_13 = shown_silhouettes_13 = np.empty(0)
    if shown_inertias_13.shape != (len(KS_13),) or shown_silhouettes_13.shape != (len(KS_13),):
        print(f"❌ Ex 15.13 : inertias_13 et silhouettes_13 doivent être deux listes de {len(KS_13)} nombres, une valeur par k de KS_13.")
    else:
        fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
        axes[0].scatter(*X_blobs_13.T, s=4, c=y_blobs_13, cmap="tab10")
        axes[0].set(title="the seven blobs (true groups)")
        axes[1].plot(KS_13, shown_inertias_13, "o-")
        axes[1].set(xlabel="k", ylabel="inertia", title="inertia")
        axes[2].plot(KS_13, shown_silhouettes_13, "o-")
        axes[2].set(xlabel="k", ylabel="mean silhouette", title="silhouette")
        plt.show()
if filled(kmeans_ari_13, dbscan_ari_13, small_eps_13, single_ari_13):   # the pictures, once d) to g) are answered
    fig, axes = plt.subplots(1, 4, figsize=(13, 3))
    for ax, (name, model) in zip(axes, [("KMeans(2)", KMeans(n_clusters=2, n_init=10, random_state=0)), ("DBSCAN(eps=0.2)", DBSCAN(eps=0.2)),
                                        ("DBSCAN(eps=0.1)", DBSCAN(eps=0.1)), ("single linkage", AgglomerativeClustering(n_clusters=2, linkage="single"))]):
        found = model.fit_predict(X_moons_13)
        ax.scatter(*X_moons_13[found >= 0].T, s=6, c=found[found >= 0], cmap="tab10")
        ax.scatter(*X_moons_13[found < 0].T, s=14, marker="x", color="black")
        ax.set(title=name, xticks=[], yticks=[])
    plt.show()

### Ex 15.14 — Transformateurs : test hors de [0, 1] et inverse_transform 📦 ★★ ⏱️ 20 min
**Objectif :** appliquer à des données nouvelles une transformation apprise sur l'entraînement, puis revenir dans les unités d'origine.  
**Prérequis :** ✏️ 15.3 · ch. 12 (min-max, règle d'or) · fiche §15.6 et §15.6.1 · **Fil rouge :** synthétique · **Parcours :** C

Deux situations du livre (§15.6, figures 15.7 à 15.12). D'abord, `TRAIN_14` et `TEST_14` : 80 points d'entraînement et 40 de test, à deux features, le test plus étendu sur la première et plus resserré sur la seconde. Ensuite, l'exemple de la circulation du ch. 12 : l'heure de la journée (`hours_14`) et le nombre de voitures par heure (`cars_14`), réunis dans `POINTS_14`. **Contrat** :
`scaler_14` : un `MinMaxScaler` appris sur `TRAIN_14` seul ; a) `test_min_14` et b) `test_max_14` : pour chaque feature, le minimum et le maximum de `TEST_14` transformé (deux listes de 2 nombres) ;
c) `outside_14` : la part des points de test (des lignes) qui sortent du carré $[0, 1] \times [0, 1]$ une fois transformés ;
`points_scaler_14` : un `MinMaxScaler` appris sur `POINTS_14` (les deux colonnes) ; d) `ends_14` : un tableau `(2, 2)`, les deux extrémités de la droite ajustée dans l'espace transformé (une `LinearRegression` des voitures transformées sur les heures transformées, prise aux heures transformées 0 et 1), ramenées dans les unités d'origine par `points_scaler_14.inverse_transform` ;
e) `slope_14` : la pente de la droite qui passe par les deux points de `ends_14`, en voitures par heure de la journée ;
f) `direct_slope_14` : la pente d'une `LinearRegression` ajustée directement sur les heures et les voitures, sans mise à l'échelle.

La vérification contrôle a) à f), puis trace ta droite sur les données d'origine.

Dans tes notes : pourquoi le test sort-il de $[0, 1]$, et est-ce un problème ? Compare e) et f) : comment expliques-tu ce que tu observes ? Qu'observerais-tu avec un `StandardScaler` ? Avec une `Ridge` à la place de la régression linéaire ?

In [ ]:
rng_14 = np.random.default_rng(1514)
TRAIN_14 = np.column_stack([rng_14.uniform(10, 30, 80), rng_14.uniform(0, 5, 80)])     # 80 training points, 2 features
TEST_14 = np.column_stack([rng_14.uniform(5, 38, 40), rng_14.uniform(1.5, 3.5, 40)])   # 40 test points
hours_14 = rng_14.uniform(6, 22, 50)                               # traffic (ch. 12): the hour of the day
cars_14 = 40 * hours_14 - 120 + rng_14.normal(0, 60, 50)           # and the number of cars per hour
POINTS_14 = np.column_stack([hours_14, cars_14])

In [ ]:
scaler_14 = ...        #    a MinMaxScaler learned on TRAIN_14 only
test_min_14 = ...      # a) list: for each feature (column), the minimum of TEST_14 transformed by scaler_14
test_max_14 = ...      # b) list: for each feature, the maximum
outside_14 = ...       # c) share of the test points (rows) that fall outside the square [0, 1] x [0, 1] once transformed
points_scaler_14 = ... #    a MinMaxScaler learned on POINTS_14 (both columns: hours and cars)
ends_14 = ...          # d) array of shape (2, 2): fit a LinearRegression of the scaled cars on the scaled hours, take its
                       #    two ends (scaled hour 0 and 1, with their predicted scaled cars), bring them back with
                       #    points_scaler_14.inverse_transform
slope_14 = ...         # e) the slope of the line through ends_14, in cars per hour of the day
direct_slope_14 = ...  # f) the slope of a LinearRegression fitted directly on hours_14 and cars_14 (no scaling)

In [ ]:
wb.check("15.14a", test_min_14)
wb.check("15.14b", test_max_14)
wb.check("15.14c", outside_14)
wb.check("15.14d", ends_14)
wb.check("15.14e", slope_14)
wb.check("15.14f", direct_slope_14)
with wb.attempt("15.14"):
    if not filled(ends_14):
        raise NotImplementedError("ends_14")
    try:
        ends_shown_14 = np.asarray(ends_14, dtype=float)
    except (TypeError, ValueError):
        ends_shown_14 = np.empty(0)
    if ends_shown_14.shape != (2, 2):
        print(f"❌ Ex 15.14 : ends_14 doit être un tableau de nombres de forme (2, 2), deux points (heure, voitures) ; reçu {type(ends_14).__name__} de forme {ends_shown_14.shape}.")
    else:
        fig, ax = plt.subplots(figsize=(6, 3.6))
        ax.scatter(hours_14, cars_14, s=12, label="traffic")
        ax.plot(*ends_shown_14.T, color="black", label="your line, back in the original units")
        ax.set(xlabel="hour of the day", ylabel="cars per hour")
        ax.legend(fontsize=8)
        plt.show()

### Ex 15.15 — PCA de 3D vers 2D et blanchiment 📦 ★★ ⏱️ 20 min
**Objectif :** réduire un nuage en trois dimensions à deux, mesurer ce que l'on perd, et voir quand le blanchiment aide le modèle suivant.  
**Prérequis :** ch. 12 (PCA, projection) · ch. 7 (k-means) · Ex 15.13 · fiche §15.7 · **Fil rouge :** synthétique · **Parcours :** M, C

L'exemple du livre (§15.7, figures 15.13 et 15.14) : un nuage en forme d'ellipsoïde aplati, en trois dimensions (`BLOB_15`, 400 points). On a caché dedans deux groupes (`group_15`), séparés le long de la **deuxième** direction de variance, beaucoup moins étendue que la première. **Contrat** :
a) `pca_15` : une `PCA(n_components=2)` entraînée sur `BLOB_15` ; `ratio_15` : sa liste `explained_variance_ratio_` ;
b) `kept_15` : la part de la variance gardée par les deux composantes ;
c) `recon_15` : l'erreur de reconstruction, la moyenne sur toutes les valeurs de $(\text{inverse\_transform}(\text{transform}(\mathbf{X})) - \mathbf{X})^2$ ;
d) `white_15` : `BLOB_15` transformé par une `PCA(n_components=2, whiten=True)` entraînée sur lui (forme `(400, 2)`) ; `white_var_15` : la variance (`ddof=1`) de chacune de ses deux colonnes ;
e) `ari_plain_15` : l'indice de Rand ajusté (15.13) entre `group_15` et les labels d'un `KMeans(n_clusters=2, n_init=10, random_state=0)` entraîné sur les deux composantes de `pca_15`, **sans** blanchiment ;
f) `ari_white_15` : le même indice, avec le k-means entraîné sur `white_15`.

La vérification contrôle a) à f), puis trace les deux projections, colorées par groupe caché.

Dans tes notes : pourquoi l'inverse de c) n'est-il pas exact ? Que fait le blanchiment aux directions, et à la forme du nuage ? Compare e) et f) : qu'est-ce qui, dans la forme des deux nuages projetés, explique ce que tu observes ? Le blanchiment aiderait-il un arbre de décision ?

In [ ]:
rng_15 = np.random.default_rng(1515)
group_15 = rng_15.integers(0, 2, 400)                     # two hidden groups (not given to the PCA nor to the k-means)
latent_15 = np.column_stack([rng_15.normal(0, 4, 400),                                       # a long direction
                             np.where(group_15 == 1, 1.2, -1.2) + rng_15.normal(0, 0.45, 400),  # the two groups
                             rng_15.normal(0, 0.3, 400)])                                    # a thin direction
rotation_15 = np.linalg.qr(rng_15.normal(size=(3, 3)))[0]
BLOB_15 = latent_15 @ rotation_15.T + np.array([5.0, -2.0, 10.0])   # a flattened, elongated blob in 3-D

In [ ]:
pca_15 = ...          # a) PCA(n_components=2) fitted on BLOB_15
ratio_15 = ...        #    list: its explained_variance_ratio_ (2 floats)
kept_15 = ...         # b) the share of the variance of BLOB_15 that the 2 components keep
recon_15 = ...        # c) the mean, over all the values, of (pca_15.inverse_transform(pca_15.transform(BLOB_15)) - BLOB_15) ** 2
white_15 = ...        # d) BLOB_15 transformed by PCA(n_components=2, whiten=True) fitted on BLOB_15, shape (400, 2)
white_var_15 = ...    #    list: the variance (ddof=1) of each column of white_15
ari_plain_15 = ...    # e) adjusted_rand_score(group_15, labels) of KMeans(n_clusters=2, n_init=10, random_state=0) fitted on
                      #    the 2 components of pca_15 (not whitened)
ari_white_15 = ...    # f) the same, with the k-means fitted on white_15

In [ ]:
wb.check("15.15a", ratio_15)
wb.check("15.15b", kept_15)
wb.check("15.15c", recon_15)
wb.check("15.15d", white_var_15)
wb.check("15.15e", ari_plain_15)
wb.check("15.15f", ari_white_15)
with wb.attempt("15.15"):
    if not filled(pca_15, white_15):
        raise NotImplementedError("pca_15, white_15")
    plain_shown_15 = pca_15.transform(BLOB_15) if hasattr(pca_15, "components_") else None
    try:
        white_shown_15 = np.asarray(white_15, dtype=float)
    except (TypeError, ValueError):
        white_shown_15 = np.empty(0)
    if plain_shown_15 is None:
        print(f"❌ Ex 15.15 : pca_15 doit être une PCA entraînée sur BLOB_15 (l'objet, pas sa projection) ; reçu {type(pca_15).__name__}.")
    elif plain_shown_15.shape != (400, 2):
        print(f"❌ Ex 15.15 : pca_15 doit garder 2 composantes (PCA(n_components=2)) ; sa projection a la forme {plain_shown_15.shape}.")
    elif white_shown_15.shape != (400, 2):
        print(f"❌ Ex 15.15 : white_15 doit être un tableau de nombres de forme (400, 2), reçu la forme {white_shown_15.shape}.")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
        for ax, data, title in zip(axes, (plain_shown_15, white_shown_15), ("PCA(2)", "PCA(2, whiten=True)")):
            ax.scatter(*data.T, s=6, c=group_15, cmap="coolwarm")
            ax.set(title=title, xlabel="component 1", ylabel="component 2", aspect="equal")
        plt.show()

### Ex 15.16 — Cinq spirales : quel ensemble les démêle ? 🔬 ★★ ⏱️ 30 min
**Objectif :** comparer des AdaBoost de modèles de base différents, compter leurs membres et trouver la bonne profondeur des arbres.  
**Prérequis :** ch. 14 (AdaBoost, SAMME, forêts) · Ex 15.9 · fiche §15.8 · **Fil rouge :** synthétique · **Parcours :** C

L'expérience du livre (§15.8, figures 15.15 à 15.17) : 1 000 points répartis en cinq spirales, une par classe, deux tiers pour l'entraînement et un tiers pour le test (`X_TRAIN_16`, `X_TEST_16`). Tous les AdaBoost ont `n_estimators=50` et `random_state=0`. **Contrat** :
a) `ridge_16` : la liste `[accuracy de test, nombre de membres gardés]` d'un AdaBoost de `RidgeClassifier()` (le nombre de membres : `len(model.estimators_)`) ;
b) `tree_16` : la même liste pour un AdaBoost de `DecisionTreeClassifier(random_state=0)`, des arbres **sans limite** de profondeur ;
c) `tree_train_16` : l'accuracy d'**entraînement** de l'AdaBoost de b) ;
d) `stumps_16` : la même liste qu'en a) pour `AdaBoostClassifier(n_estimators=50, random_state=0)`, avec ses membres par défaut ;
e) `depth_curve_16` : pour chaque `max_depth` de `DEPTHS_16`, la liste `[accuracy de test, membres gardés]` d'un AdaBoost de `DecisionTreeClassifier(max_depth=depth, random_state=0)` (une liste de listes) ;
f) `forest_16` : l'accuracy de test d'une `RandomForestClassifier(n_estimators=100, random_state=0)`.

La vérification contrôle a) à f), trace la courbe de e), puis les frontières de trois AdaBoost.

Dans tes notes : combien de membres les AdaBoost de a), b) et d) ont-ils gardés, et pourquoi ce nombre-là ? Que dit l'accuracy des souches de ce qu'une souche peut découper ? Quelle profondeur choisirais-tu, et comment, sans regarder le test ?

In [ ]:
X_16, y_16 = wb.synth.make_spirals(n=1000, n_classes=5, noise=0.2, seed=1516)   # five spirals, one per class
X_TRAIN_16, X_TEST_16, Y_TRAIN_16, Y_TEST_16 = train_test_split(X_16, y_16, test_size=1 / 3, random_state=1516, stratify=y_16)
DEPTHS_16 = [1, 2, 3, 4, 6, 8, 12, None]

In [ ]:
ridge_16 = ...         # a) list [test accuracy, number of members kept] of
                       #    AdaBoostClassifier(estimator=RidgeClassifier(), n_estimators=50, random_state=0)
tree_16 = ...          # b) the same list for AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=0),
                       #    n_estimators=50, random_state=0)  (trees without depth limit)
tree_train_16 = ...    # c) the training accuracy of the AdaBoost of b)
stumps_16 = ...        # d) the same list as a) for AdaBoostClassifier(n_estimators=50, random_state=0) (its default members)
depth_curve_16 = ...   # e) list: for each max_depth of DEPTHS_16, [test accuracy, members kept] of the AdaBoost of trees
                       #    DecisionTreeClassifier(max_depth=depth, random_state=0), n_estimators=50, random_state=0
forest_16 = ...        # f) the test accuracy of RandomForestClassifier(n_estimators=100, random_state=0)

In [ ]:
wb.check("15.16a", ridge_16)
wb.check("15.16b", tree_16)
wb.check("15.16c", tree_train_16)
wb.check("15.16d", stumps_16)
wb.check("15.16e", depth_curve_16)
wb.check("15.16f", forest_16)
with wb.attempt("15.16"):
    if not filled(depth_curve_16):
        raise NotImplementedError("depth_curve_16")
    try:
        curve_16 = np.asarray(depth_curve_16, dtype=float)
    except (TypeError, ValueError):
        curve_16 = np.empty(0)
    if curve_16.shape != (len(DEPTHS_16), 2):
        print(f"❌ Ex 15.16 : depth_curve_16 doit être une liste de {len(DEPTHS_16)} paires [accuracy, membres], une par profondeur de DEPTHS_16.")
    else:
        labels_16 = [str(d) for d in DEPTHS_16]
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
        axes[0].plot(labels_16, curve_16[:, 0], "o-")
        axes[0].set(xlabel="max_depth of the members", ylabel="test accuracy")
        axes[1].plot(labels_16, curve_16[:, 1], "s-")
        axes[1].set(xlabel="max_depth of the members", ylabel="members kept (n_estimators=50)")
        plt.show()
if filled(ridge_16, tree_16, stumps_16):        # the frontiers, once a), b) and d) are answered
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
    for ax, (name, model) in zip(axes, [("AdaBoost of RidgeClassifier", AdaBoostClassifier(estimator=RidgeClassifier(), n_estimators=50, random_state=0)),
                                        ("AdaBoost of stumps", AdaBoostClassifier(n_estimators=50, random_state=0)),
                                        ("AdaBoost of unlimited trees", AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=0), n_estimators=50, random_state=0))]):
        DecisionBoundaryDisplay.from_estimator(model.fit(X_TRAIN_16, Y_TRAIN_16), X_16, response_method="predict",
                                               alpha=0.4, cmap="tab10", ax=ax, grid_resolution=150)
        ax.scatter(*X_TEST_16.T, c=Y_TEST_16, s=5, cmap="tab10", vmin=0, vmax=9)
        ax.set(title=name, xticks=[], yticks=[])
    plt.show()

## Partie C · Évaluer, régler et enchaîner

La validation croisée et les estimateurs `...CV` (15.17), les recherches en grille et au hasard (15.18 à 15.20), la frontière de décision (15.21), le pipeline polynomial et la lecture de `cv_results_` (15.22 et 15.23), puis la fuite du ch. 8 refaite avec un pipeline (15.24). La cellule suivante importe les distributions de SciPy et la sélection de features.

In [ ]:
from scipy.stats import loguniform, randint, uniform
from sklearn.feature_selection import SelectKBest, f_classif
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3-D projection)

### Ex 15.17 — cross_validate, KFold, StratifiedKFold et RidgeClassifierCV 📦 ★★ ⏱️ 25 min
**Objectif :** choisir ses folds, lire plusieurs mesures à la fois, et comprendre ce que fait vraiment un estimateur `...CV`.  
**Prérequis :** ch. 8 (validation croisée, stratification) · Ex 15.9 · fiche §15.9.1 · **Fil rouge :** Penguins · **Parcours :** R, M, C

Les 333 manchots, **dans l'ordre du fichier** (triés par espèce : 146 Adélie, puis 119 Gentoo, puis 68 Chinstrap), décrits par deux mesures : l'épaisseur du bec et la nageoire (`X_17`). `MODEL_17` : une régression logistique après un `StandardScaler`. **Contrat** :
a) `kfold_17` : la liste des 5 scores de `cross_val_score(MODEL_17, X_17, Y_17, cv=KFold(5))` (sans mélange) ;
b) `default_17` : la moyenne des scores avec `cv=5` ;
`results_17` : `cross_validate(MODEL_17, X_17, Y_17, cv=FOLDS_17, scoring=["accuracy", "f1_macro"], return_train_score=True)`, avec `FOLDS_17` un `StratifiedKFold` mélangé ; c) `test_acc_17`, d) `train_acc_17` et e) `f1_17` : les moyennes de l'accuracy de validation, de l'accuracy d'entraînement et du F1 macro de validation ;
`ridge_cv_17` : un pipeline `PolynomialFeatures(degree=3)`, `StandardScaler`, puis `RidgeClassifierCV(alphas=ALPHAS_17)`, entraîné sur `X_17` et `Y_17` ; f) `alpha_17` : l'`alpha` qu'il a choisi ; g) `score_17` : `ridge_cv_17.score(X_17, Y_17)` ;
h) `outer_17` : la moyenne de `cross_val_score` d'un **nouveau** pipeline identique à `ridge_cv_17`, avec `cv=FOLDS_17`.

La vérification contrôle a) à h).

Dans tes notes : que s'est-il passé dans le dernier fold de a) ? Pourquoi b) ne souffre-t-il pas du tri, alors que `cv=5` ne mélange pas ? Qu'est-ce que `RidgeClassifierCV` a fait par validation croisée, et lequel de g) et h) estime honnêtement sa performance ?

In [ ]:
X_17 = PENGUINS[["bill_depth_mm", "flipper_length_mm"]].to_numpy()   # two measures, the 333 penguins
Y_17 = PENGUINS["species"].to_numpy()                                    # in the order of the file: sorted by species
MODEL_17 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
FOLDS_17 = StratifiedKFold(n_splits=5, shuffle=True, random_state=1517)
ALPHAS_17 = [0.01, 0.1, 1, 10, 100]
print("the first, middle and last penguins:", Y_17[0], Y_17[166], Y_17[-1])

In [ ]:
kfold_17 = ...      # a) list: the 5 scores of cross_val_score(MODEL_17, X_17, Y_17, cv=KFold(5))  (no shuffling)
default_17 = ...    # b) the mean of cross_val_score(MODEL_17, X_17, Y_17, cv=5)
results_17 = ...    #    cross_validate(MODEL_17, X_17, Y_17, cv=FOLDS_17, scoring=["accuracy", "f1_macro"],
                    #    return_train_score=True)
test_acc_17 = ...   # c) the mean of its validation accuracies (key "test_accuracy")
train_acc_17 = ...  # d) the mean of its training accuracies
f1_17 = ...         # e) the mean of its validation F1 (macro)
ridge_cv_17 = ...   #    make_pipeline(PolynomialFeatures(degree=3), StandardScaler(), RidgeClassifierCV(alphas=ALPHAS_17)),
                    #    fitted on X_17, Y_17
alpha_17 = ...      # f) the alpha it chose
score_17 = ...      # g) ridge_cv_17.score(X_17, Y_17)
outer_17 = ...      # h) the mean of cross_val_score of a NEW pipeline, identical to ridge_cv_17, with cv=FOLDS_17

In [ ]:
wb.check("15.17a", kfold_17)
wb.check("15.17b", default_17)
wb.check("15.17c", test_acc_17)
wb.check("15.17d", train_acc_17)
wb.check("15.17e", f1_17)
wb.check("15.17f", alpha_17)
wb.check("15.17g", score_17)
wb.check("15.17h", outer_17)

### Ex 15.18 — GridSearchCV : best_params_, cv_results_ et score d'entraînement 📦 ★★ ⏱️ 25 min
**Objectif :** lancer une recherche en grille et lire tout ce qu'elle rend, égalités comprises.  
**Prérequis :** ✏️ 15.1 · Ex 15.17 · fiche §15.9.2 et §15.9.3 · **Fil rouge :** synthétique · **Parcours :** R, M, C

L'exemple du livre (§15.9.3) : un `RidgeClassifier` sur deux demi-lunes bruitées, réglé sur six valeurs d'`alpha` et trois algorithmes de calcul (`solver`), soit 18 candidats. Les 400 points sont coupés en 300 pour la recherche et 100 pour le test. **Contrat** :
`search_18` : `GridSearchCV(RidgeClassifier(), GRID_18, cv=FOLDS_18, return_train_score=True)`, entraîné sur `X_TRAIN_18`, `Y_TRAIN_18` ; a) `best_alpha_18` et b) `best_solver_18` : le réglage du meilleur candidat ;
c) `best_score_18` : son score moyen de validation ;
d) `n_candidates_18` : le nombre de candidats évalués ;
e) `ties_18` : le nombre de candidats classés premiers (`rank_test_score` égal à 1) ;
f) `second_solver_18` : le `solver` du **deuxième** candidat, dans l'ordre de `cv_results_["params"]` ;
g) `gap_18` : pour le meilleur candidat, son score moyen d'entraînement moins son score moyen de validation ;
h) `test_18` : l'accuracy de `search_18` sur le test, mesurée une seule fois.

La vérification contrôle a) à h), puis affiche les premières lignes de `cv_results_`.

Dans tes notes : pourquoi tant d'égalités, et pourquoi le `solver` ne change-t-il rien ici ? Comment la recherche départage-t-elle des candidats à égalité ? Que dit `gap_18` de l'overfitting ? Pourquoi `test_18` et `best_score_18` diffèrent-ils ?

In [ ]:
X_18, Y_18 = wb.synth.make_moons(n=400, noise=0.3, seed=1518)
X_TRAIN_18, X_TEST_18, Y_TRAIN_18, Y_TEST_18 = train_test_split(X_18, Y_18, test_size=0.25, random_state=1518, stratify=Y_18)
GRID_18 = {"solver": ["svd", "cholesky", "lsqr"], "alpha": [0.001, 0.01, 0.1, 1, 10, 100]}   # 18 candidates
FOLDS_18 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
search_18 = ...        #    GridSearchCV(RidgeClassifier(), GRID_18, cv=FOLDS_18, return_train_score=True), fitted on
                       #    X_TRAIN_18, Y_TRAIN_18
best_alpha_18 = ...    # a) the alpha of the best candidate
best_solver_18 = ...   # b) its solver (a string)
best_score_18 = ...    # c) the mean validation score of the best candidate
n_candidates_18 = ...  # d) the number of candidates evaluated
ties_18 = ...          # e) the number of candidates ranked first (rank_test_score equal to 1)
second_solver_18 = ... # f) the solver of the SECOND candidate, in the order of cv_results_["params"]
gap_18 = ...           # g) mean_train_score minus mean_test_score, for the best candidate
test_18 = ...          # h) the accuracy of search_18 on the test part (X_TEST_18, Y_TEST_18), measured once

In [ ]:
wb.check("15.18a", best_alpha_18)
wb.check("15.18b", best_solver_18)
wb.check("15.18c", best_score_18)
wb.check("15.18d", n_candidates_18)
wb.check("15.18e", ties_18)
wb.check("15.18f", second_solver_18)
wb.check("15.18g", gap_18)
wb.check("15.18h", test_18)
with wb.attempt("15.18"):
    if not filled(search_18):
        raise NotImplementedError("search_18")
    shown_18 = pd.DataFrame(search_18.cv_results_)[["param_alpha", "param_solver", "mean_test_score", "std_test_score",
                                                    "rank_test_score"]]
    print(shown_18.head(8).to_string())

### Ex 15.19 — RandomizedSearchCV avec des distributions 📦 ★★ ⏱️ 25 min
**Objectif :** régler un modèle par une recherche au hasard, choisir ses distributions et savoir quand elle tire avec remise.  
**Prérequis :** Ex 15.18 · fiche §15.9.4 (encadré sur les distributions) · **Fil rouge :** synthétique · **Parcours :** M, C

Un `SVC` à noyau RBF sur 400 points de deux demi-lunes, réglé par une recherche au hasard sur `C` et `gamma`. **Contrat** :
`random_19` : `RandomizedSearchCV(SVC(), {"C": loguniform(1e-2, 1e3), "gamma": loguniform(1e-3, 1e2)}, n_iter=20, cv=FOLDS_19, random_state=1519)`, entraîné sur `X_19`, `Y_19` ; a) `best_C_19` et b) `best_gamma_19` : le réglage du meilleur candidat (deux `float`) ;
c) `best_score_19` : son score moyen de validation ;
d) `distinct_C_19` : le nombre de valeurs **distinctes** de `C` parmi les 20 candidats ;
e) `list_dupes_19` : pour `RandomizedSearchCV(KNeighborsClassifier(), {"n_neighbors": list(range(1, 31)), "weights": ["uniform", "distance"]}, n_iter=20, cv=FOLDS_19, random_state=1519)` entraîné sur les mêmes données, le nombre de candidats qui **répètent** un candidat précédent (20 moins le nombre de candidats distincts) ;
f) `randint_dupes_19` : le même nombre, avec `"n_neighbors": randint(1, 31)` à la place de la liste ;
g) `log_share_19` : la part des valeurs inférieures à 1 parmi `loguniform(1e-3, 1e3).rvs(10_000, random_state=1519)` ;
h) `uniform_share_19` : la même part pour `uniform(1e-3, 1e3).rvs(10_000, random_state=1519)`.

La vérification contrôle a) à h), puis place les 20 candidats de a) sur des axes logarithmiques, colorés par leur score.

Dans tes notes : pourquoi une échelle logarithmique pour `C` et `gamma` ? Que tire `uniform(1e-3, 1e3)` au juste (relis l'encadré sur ses deux arguments) ? Compare g) et h), puis e) et f) : qu'est-ce qui explique ce que tu observes, et qu'est-ce que cela coûte ?

In [ ]:
X_19, Y_19 = wb.synth.make_moons(n=400, noise=0.3, seed=1519)
FOLDS_19 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
random_19 = ...       #    RandomizedSearchCV(SVC(), {"C": loguniform(1e-2, 1e3), "gamma": loguniform(1e-3, 1e2)},
                      #    n_iter=20, cv=FOLDS_19, random_state=1519), fitted on X_19, Y_19
best_C_19 = ...       # a) the C of the best candidate (a float)
best_gamma_19 = ...   # b) its gamma
best_score_19 = ...   # c) its mean validation score
distinct_C_19 = ...   # d) the number of distinct values of C among the 20 candidates
list_dupes_19 = ...   # e) RandomizedSearchCV(KNeighborsClassifier(), {"n_neighbors": list(range(1, 31)),
                      #    "weights": ["uniform", "distance"]}, n_iter=20, cv=FOLDS_19, random_state=1519) fitted on X_19, Y_19:
                      #    the number of candidates that repeat an earlier candidate (20 minus the number of distinct ones)
randint_dupes_19 = ...# f) the same count with "n_neighbors": randint(1, 31) instead of the list
log_share_19 = ...    # g) the share of the values below 1 among loguniform(1e-3, 1e3).rvs(10_000, random_state=1519)
uniform_share_19 = ...# h) the same share for uniform(1e-3, 1e3).rvs(10_000, random_state=1519)

In [ ]:
wb.check("15.19a", best_C_19)
wb.check("15.19b", best_gamma_19)
wb.check("15.19c", best_score_19)
wb.check("15.19d", distinct_C_19)
wb.check("15.19e", list_dupes_19)
wb.check("15.19f", randint_dupes_19)
wb.check("15.19g", log_share_19)
wb.check("15.19h", uniform_share_19)
with wb.attempt("15.19"):
    if not filled(random_19):
        raise NotImplementedError("random_19")
    tried_19 = pd.DataFrame(random_19.cv_results_)
    fig, ax = plt.subplots(figsize=(5.5, 4))
    points_19 = ax.scatter(tried_19["param_C"].astype(float), tried_19["param_gamma"].astype(float),
                           c=tried_19["mean_test_score"], cmap="viridis", s=40)
    ax.set(xscale="log", yscale="log", xlabel="C", ylabel="gamma", title="20 candidates drawn on log scales")
    fig.colorbar(points_19, label="mean validation accuracy")
    plt.show()

### Ex 15.20 — Grille ou hasard : qui gagne avec 16 essais ? 🔮 ★★ ⏱️ 20 min
**Objectif :** prévoir, avant de le simuler, quand 16 essais au hasard battent une grille de 4 × 4, et le justifier par le calcul.  
**Prérequis :** Ex 15.19 · 🧮 15.5 · fiche §15.9.2 (Bergstra et Bengio, la formule $1 - (1 - q)^{n}$) · **Fil rouge :** synthétique · **Parcours :** M, C

Pour aller vite, pas de modèle à entraîner : trois **surfaces de validation inventées** sur le carré $[0, 1] \times [0, 1]$ de deux hyperparamètres $h_1$ et $h_2$, connues d'avance (`surface_a_20`, `surface_b_20`, `surface_c_20`). Une grille de $4 \times 4$ (les valeurs 0,125, 0,375, 0,625 et 0,875 de chaque hyperparamètre) et 16 essais tirés uniformément dans le carré coûtent autant ; on compare le meilleur score de chacun. Les trois surfaces :
- a) seul $h_1$ compte : un pic étroit en $h_1 = 0{,}53$ ;
- b) les deux comptent : un pic rond en $(0{,}53 ;\ 0{,}47)$ ;
- c) les deux comptent, et le pic tombe **exactement** sur un point de la grille, $(0{,}375 ;\ 0{,}625)$.

Prédis, **avant** d'exécuter quoi que ce soit, pour chaque surface, la part des 2 000 graines où les 16 essais au hasard battent **strictement** la grille : `"A"` moins de 25 % ; `"B"` de 25 à 50 % ; `"C"` de 50 à 75 % ; `"D"` plus de 75 %. Trois prédictions : `share_a_20`, `share_b_20` et `share_c_20`.

Puis exécute l'expérience : elle compte les victoires sur 2 000 graines et trace les trois surfaces, la grille et un tirage de 16 essais. Après l'expérience, la question d) te fait retrouver un de ces nombres par le calcul.

In [ ]:
GRID_20 = np.array([0.125, 0.375, 0.625, 0.875])        # 4 values per hyperparameter: a 4 x 4 grid of 16 points


def surface_a_20(h1, h2):
    """A made-up validation score on [0, 1] x [0, 1]: only h1 matters (a narrow peak at h1 = 0.53)."""
    return np.exp(-(h1 - 0.53) ** 2 / (2 * 0.05 ** 2)) + 0 * h2


def surface_b_20(h1, h2):
    """Both hyperparameters matter: a round peak at (0.53, 0.47)."""
    return np.exp(-((h1 - 0.53) ** 2 + (h2 - 0.47) ** 2) / (2 * 0.1 ** 2))


def surface_c_20(h1, h2):
    """Both matter, and the peak sits exactly on a point of the grid, (0.375, 0.625)."""
    return np.exp(-((h1 - 0.375) ** 2 + (h2 - 0.625) ** 2) / (2 * 0.1 ** 2))


def grid_best_20(surface):
    """The best score of the 16 points of the grid."""
    h1, h2 = np.meshgrid(GRID_20, GRID_20)
    return float(surface(h1, h2).max())


def random_best_20(surface, seed):
    """The best score of 16 points drawn uniformly in the square, with np.random.default_rng(seed)."""
    points = np.random.default_rng(seed).random((16, 2))
    return float(surface(points[:, 0], points[:, 1]).max())

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
share_a_20 = ...   # surface a: "A", "B", "C" or "D"
share_b_20 = ...   # surface b: "A", "B", "C" or "D"
share_c_20 = ...   # surface c: "A", "B", "C" or "D"

In [ ]:
if any(answer is ... or answer is None for answer in [share_a_20, share_b_20, share_c_20]):
    print("⏳ Ex 15.20 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    for name, surface in [("a", surface_a_20), ("b", surface_b_20), ("c", surface_c_20)]:
        wins = np.mean([random_best_20(surface, seed) > grid_best_20(surface) for seed in range(2000)])
        print(f"surface {name}: 16 random trials beat the 4 x 4 grid for {wins:.1%} of 2,000 seeds "
              f"(grid best {grid_best_20(surface):.3f})")
    h1_20, h2_20 = np.meshgrid(np.linspace(0, 1, 201), np.linspace(0, 1, 201))
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
    drawn_20 = np.random.default_rng(0).random((16, 2))
    for ax, (name, surface) in zip(axes, [("a", surface_a_20), ("b", surface_b_20), ("c", surface_c_20)]):
        ax.contourf(h1_20, h2_20, surface(h1_20, h2_20), levels=20, cmap="viridis")
        grid_h1_20, grid_h2_20 = np.meshgrid(GRID_20, GRID_20)
        ax.scatter(grid_h1_20, grid_h2_20, marker="s", color="white", edgecolor="black", s=30, label="grid")
        ax.scatter(*drawn_20.T, marker="o", color="tab:red", s=18, label="16 random trials (seed 0)")
        ax.set(title=f"surface {name}", xlabel="h1", ylabel="h2", aspect="equal")
    axes[0].legend(fontsize=7, loc="lower left")
    plt.show()
    wb.check("15.20a", share_a_20)
    wb.check("15.20b", share_b_20)
    wb.check("15.20c", share_c_20)

**d)** Pour la surface b, les 16 essais battent la grille quand l'un d'eux tombe **plus près du pic** que le meilleur point de la grille. Calcule la probabilité exacte de cet événement (la formule $1 - (1 - q)^{n}$ de la fiche, avec $q$ la part du carré concernée ; ce disque est tout entier dans le carré), et compare-la à la simulation.

Dans tes notes : compare avec tes prédictions, et explique chacun des trois résultats par ce que la grille et le hasard essaient sur chaque axe. Une vraie surface de validation ressemble-t-elle plutôt à a), à b) ou à c) ? Que t'apprend le cas c) sur les comparaisons « grille contre hasard » ?

In [ ]:
theory_b_20 = ...   # d) for surface b, the exact probability that at least one of 16 uniform random trials falls closer to
                    #    the peak (0.53, 0.47) than the best point of the grid (all of this disc lies inside the square)

In [ ]:
wb.check("15.20d", theory_b_20)

### Ex 15.21 — Frontières de décision : DecisionBoundaryDisplay et surfaces 3D 📦 ★★ ⏱️ 20 min
**Objectif :** tracer la frontière et la surface de score d'un classifieur à deux features, en quelques lignes.  
**Prérequis :** Ex 15.10 · Ex 15.18 · fiche §15.9.6 · **Fil rouge :** synthétique · **Parcours :** C

`MODEL_21` est déjà entraîné : un pipeline `PolynomialFeatures(degree=3)`, `StandardScaler`, `RidgeClassifier(alpha=1.0)` sur 300 points de deux demi-lunes. Les figures du livre (§15.9.6, figures 15.36 et 15.37) demandaient une grille et une boucle ; `DecisionBoundaryDisplay.from_estimator(modèle, X, response_method=...)` calcule la grille et trace d'un coup, puis garde ses calculs dans ses attributs `xx0`, `xx1` (la grille) et `response` (les valeurs tracées), et ses axes dans `ax_`. **Contrat** :
a) `classes_21` : le `DecisionBoundaryDisplay` des classes prédites (`response_method="predict"`), avec les points de `X_21` dessinés par-dessus (`classes_21.ax_.scatter(...)`) ;
b) `scores_21` : celui du score (`response_method="decision_function"`, une palette divergente comme `"RdBu_r"`), avec la frontière, le niveau 0 du score, tracée par `scores_21.ax_.contour(scores_21.xx0, scores_21.xx1, scores_21.response, levels=[0])` ;
c) dans une cellule à toi : la surface 3D du score, écrêtée à $[-1, 1]$ comme dans le livre (`np.clip`), avec `ax = plt.figure().add_subplot(projection="3d")` et `ax.plot_surface(...)`.

La vérification contrôle que a) montre des classes et b) le score de `decision_function` ; les figures se corrigent avec `05_solutions.md`.

Dans tes notes : où la frontière de a) se trouve-t-elle sur la carte de b) ? Pourquoi le livre écrête-t-il la surface ? Que montrerait `response_method="predict_proba"` sur ce modèle ?

In [ ]:
X_21, Y_21 = wb.synth.make_moons(n=300, noise=0.25, seed=1521)
MODEL_21 = make_pipeline(PolynomialFeatures(degree=3), StandardScaler(), RidgeClassifier(alpha=1.0)).fit(X_21, Y_21)

In [ ]:
classes_21 = ...  # a) DecisionBoundaryDisplay.from_estimator(MODEL_21, X_21, response_method="predict", ...) : the classes,
                  #    with the points of X_21 drawn on its axes (classes_21.ax_)
scores_21 = ...   # b) the same with response_method="decision_function" (a diverging colour map), plus the frontier:
                  #    the level 0 of the score (ax.contour(scores_21.xx0, scores_21.xx1, scores_21.response, levels=[0]))
# c) in a cell of your own: a 3-D surface of the score (scores_21.xx0, scores_21.xx1, scores_21.response), clipped to
#    [-1, 1] like the book (np.clip), with ax = plt.figure().add_subplot(projection="3d") and ax.plot_surface(...)

In [ ]:
with checked("15.21"):
    if not filled(classes_21, scores_21):
        raise NotImplementedError("classes_21, scores_21")
    displays_ok_21 = isinstance(classes_21, DecisionBoundaryDisplay) and isinstance(scores_21, DecisionBoundaryDisplay)
    verdict("15.21", displays_ok_21, "deux DecisionBoundaryDisplay.",
            "classes_21 et scores_21 doivent être les objets rendus par DecisionBoundaryDisplay.from_estimator.")
    if displays_ok_21:
        grid_21 = np.column_stack([scores_21.xx0.ravel(), scores_21.xx1.ravel()])
        verdict("15.21", np.array_equal(np.unique(classes_21.response), [0, 1]),
                "a) montre les classes prédites (0 et 1).", "a) doit montrer predict : des classes 0 et 1.")
        same_21 = np.allclose(np.ravel(scores_21.response), MODEL_21.decision_function(grid_21))
        verdict("15.21", same_21, "b) montre le score de decision_function.",
                "b) doit montrer le score de decision_function (response_method=\"decision_function\").")
        if same_21:
            agree_21 = np.mean(MODEL_21.predict(grid_21) == (np.ravel(scores_21.response) > 0))
            print(f"on the {len(grid_21)} points of the grid, predict gives class 1 exactly where the score is "
                  f"positive: {agree_21:.0%}")

### Ex 15.22 — Pipeline polynomial + ridge : degré et alpha ensemble 📦 ★★★ ⏱️ 35 min
**Objectif :** régler ensemble le degré des features polynomiales et la régularisation, dans un pipeline et une recherche en grille.  
**Prérequis :** Ex 15.18 · ✏️ 15.4 · ch. 9 (features polynomiales, régularisation) · fiche §15.9.5 · **Fil rouge :** synthétique · **Parcours :** R, C

L'exemple du livre (§15.9.5) : un `RidgeClassifier` ne trace qu'une droite ; avec des features polynomiales, sa frontière se courbe. 400 points de deux demi-lunes bruitées, 300 pour la recherche et 100 pour le test. **Contrat** :
`pipe_22` : un `Pipeline` de trois étapes nommées `"poly"` (`PolynomialFeatures()`), `"scale"` (`StandardScaler()`) et `"ridge"` (`RidgeClassifier()`) ; `grid_22` : la grille des degrés `DEGREES_22` et des alphas `ALPHAS_22`, avec les noms des étapes ; `search_22` : `GridSearchCV(pipe_22, grid_22, cv=FOLDS_22, return_train_score=True)`, entraîné sur la partie recherche ;
a) `best_22` : la liste `[meilleur degré, meilleur alpha]`, lue dans `best_params_` ;
b) `best_score_22` : le score moyen de validation du meilleur candidat ;
c) `n_features_22` : le nombre de colonnes que produit l'étape `"poly"` du meilleur pipeline (son attribut `n_output_features_`) ;
d) `test_22` : l'accuracy de `search_22` sur le test, mesurée une seule fois ;
e) `degree0_22` : le score moyen de validation des candidats de degré 0 (tous ont le même) ;
f) `overfit_22` : pour le degré 9 et l'alpha 0,001, le score moyen d'entraînement moins le score moyen de validation.

La vérification contrôle a) à f), puis trace les frontières du degré 1, du meilleur degré et du degré 9.

Dans tes notes : retrouve c) avec la formule de ✏️ 15.4. Pourquoi le degré 0 vaut-il ce score ? Le meilleur réglage est-il au bord de la grille, et que faudrait-il faire s'il l'était ? Pourquoi le `StandardScaler` est-il **après** `PolynomialFeatures` ?

In [ ]:
X_22, Y_22 = wb.synth.make_moons(n=400, noise=0.3, seed=1522)
X_TRAIN_22, X_TEST_22, Y_TRAIN_22, Y_TEST_22 = train_test_split(X_22, Y_22, test_size=0.25, random_state=1522, stratify=Y_22)
FOLDS_22 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
DEGREES_22 = [0, 1, 2, 3, 5, 7, 9]
ALPHAS_22 = [0.001, 0.1, 10, 1000]

In [ ]:
pipe_22 = ...          #    Pipeline([("poly", PolynomialFeatures()), ("scale", StandardScaler()), ("ridge", RidgeClassifier())])
grid_22 = ...          #    the grid of the degrees DEGREES_22 and the alphas ALPHAS_22, with the names of the steps
search_22 = ...        #    GridSearchCV(pipe_22, grid_22, cv=FOLDS_22, return_train_score=True), fitted on X_TRAIN_22, Y_TRAIN_22
best_22 = ...          # a) list [best degree, best alpha], read in search_22.best_params_
best_score_22 = ...    # b) the mean validation score of the best candidate
n_features_22 = ...    # c) the number of columns that the poly step of the best pipeline produces (n_output_features_)
test_22 = ...          # d) the accuracy of search_22 on the test part, measured once
degree0_22 = ...       # e) the mean validation score of the candidates of degree 0 (they all have the same)
overfit_22 = ...       # f) for degree 9 and alpha 0.001: mean_train_score minus mean_test_score

In [ ]:
wb.check("15.22a", best_22)
wb.check("15.22b", best_score_22)
wb.check("15.22c", n_features_22)
wb.check("15.22d", test_22)
wb.check("15.22e", degree0_22)
wb.check("15.22f", overfit_22)
with wb.attempt("15.22"):
    if not filled(search_22):
        raise NotImplementedError("search_22")
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
    for ax, degree in zip(axes, (1, search_22.best_params_["poly__degree"], 9)):
        model_22 = clone(search_22.best_estimator_).set_params(poly__degree=degree).fit(X_TRAIN_22, Y_TRAIN_22)
        DecisionBoundaryDisplay.from_estimator(model_22, X_22, response_method="predict", alpha=0.35, cmap="coolwarm", ax=ax)
        ax.scatter(*X_TRAIN_22.T, c=Y_TRAIN_22, s=6, cmap="coolwarm")
        ax.set(title=f"degree {degree}, alpha {search_22.best_params_['ridge__alpha']}", xticks=[], yticks=[])
    plt.show()

### Ex 15.23 — Lire cv_results_ : carte de chaleur et écarts-types 📈 ★★ ⏱️ 20 min
**Objectif :** transformer `cv_results_` en tableau et en carte de chaleur, et y lire ce qu'une recherche ne dit pas seule.  
**Prérequis :** Ex 15.22 · MP2 (règle d'une erreur type) · fiche §15.9.3 et §15.9.5 · **Fil rouge :** synthétique · **Parcours :** C

`SEARCH_23` est une recherche déjà faite : le pipeline polynomial de 15.22, sur deux cercles emboîtés bruités, avec 7 degrés et 4 alphas (28 candidats). **Contrat** :
a) `table_23` : un tableau d'une ligne par degré et d'une colonne par alpha, rempli par `mean_test_score` (`pd.DataFrame(SEARCH_23.cv_results_).pivot(...)`), et `spread_23`, le même pour `std_test_score` ; trace-les en cartes de chaleur (`plt.imshow`), avec les nombres dans les cases ;
b) `best_degree_23` : la liste, pour chaque alpha (dans l'ordre croissant), du degré qui a le meilleur score moyen dans sa colonne (à égalité, le plus petit degré, comme le rend `idxmax`) ;
c) `below_23` : le nombre de candidats dont le score moyen est **sous** 0,5, le score du modèle constant de degré 0 ;
d) `close_23` : le nombre de candidats dont le score moyen atteint au moins `best_score_` moins **une erreur type** du meilleur candidat (son `std_test_score` divisé par la racine du nombre de folds, 5).

La vérification contrôle a) à d).

Dans tes notes : en lisant tes cartes, que fait une forte régularisation au meilleur degré ? Pourquoi le degré 1 fait-il moins bien que le modèle constant sur des cercles ? Selon la règle d'une erreur type, choisirais-tu un autre réglage que le meilleur ? Où les écarts-types sont-ils les plus grands, et qu'en conclus-tu sur les différences de la carte des moyennes ?

In [ ]:
X_23, Y_23 = make_circles(n_samples=300, noise=0.15, factor=0.5, random_state=1523)   # two noisy nested circles
pipe_23 = Pipeline([("poly", PolynomialFeatures()), ("scale", StandardScaler()), ("ridge", RidgeClassifier())])
SEARCH_23 = GridSearchCV(pipe_23, {"poly__degree": [0, 1, 2, 3, 4, 6, 8], "ridge__alpha": [0.01, 1, 100, 10000]},
                         cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0)).fit(X_23, Y_23)
print("the search is done:", len(SEARCH_23.cv_results_["params"]), "candidates")

In [ ]:
table_23 = ...        # a) a DataFrame of SEARCH_23.cv_results_, pivoted: one row per degree, one column per alpha, the
                      #    values of mean_test_score (pandas: pivot); draw it as a heat map (plt.imshow), with its numbers
spread_23 = ...       #    the same table for std_test_score
best_degree_23 = ...  # b) list: for each alpha (in increasing order), the degree of the best mean score in that column
                      #    (in case of a tie, the smallest degree, as idxmax returns it)
below_23 = ...        # c) the number of candidates whose mean score is BELOW 0.5, the score of the constant model of degree 0
close_23 = ...        # d) the number of candidates whose mean score is at least best_score_ minus ONE standard error of the
                      #    best candidate (its std_test_score divided by the square root of the number of folds, 5)

In [ ]:
with wb.attempt("15.23"):
    if not filled(table_23):
        raise NotImplementedError("table_23")
    try:
        values_23 = np.asarray(table_23, dtype=float)
    except (TypeError, ValueError):
        values_23 = None
    if values_23 is None or values_23.shape != (7, 4):
        print(f"❌ Ex 15.23 : table_23 doit être un tableau de 7 degrés (lignes) × 4 alphas (colonnes) ; "
              f"reçu {None if values_23 is None else values_23.shape}.")
    else:
        wb.check("15.23a", values_23)
wb.check("15.23b", best_degree_23)
wb.check("15.23c", below_23)
wb.check("15.23d", close_23)

### Ex 15.24 — Refaire l'expérience de fuite du ch. 8 (sélection de features sur du bruit) avec un Pipeline 🔬 ★★ ⏱️ 25 min
**Objectif :** mesurer la fuite d'une sélection de features faite avant la validation croisée, et voir un pipeline l'empêcher.  
**Prérequis :** Ex 15.22 · 🔬 8.25 · fiche §15.9.7 · **Fil rouge :** synthétique · **Parcours :** R, C

Au ch. 8 (🔬 8.25), tu as écrit toi-même la boucle qui sélectionne les features dans chaque fold. Ici, scikit-learn le fait. Données : 200 exemples, 2 000 features de **bruit pur**, des labels tirés au hasard : aucun modèle ne peut faire mieux que 50 % en moyenne. `SelectKBest(f_classif, k=...)` garde les $k$ features les plus liées à la cible (un test statistique par feature ; elle regarde donc `y`). **Contrat** :
a) `leaky_24` : `SelectKBest(f_classif, k=20)` entraîné sur **toutes** les lignes, puis la moyenne de `cross_val_score(LogisticRegression(max_iter=1000), les 20 colonnes gardées, Y_24, cv=FOLDS_24)` ;
b) `honest_24` : la moyenne de `cross_val_score` du pipeline `SelectKBest(f_classif, k=20)` puis `LogisticRegression(max_iter=1000)`, sur `X_24`, avec `cv=FOLDS_24` ;
c) `search_24` : une `GridSearchCV` de ce pipeline sur `selectkbest__k` dans `KS_24`, avec `cv=FOLDS_24`, entraînée sur `X_24`, `Y_24` ; `inner_24` : son `best_score_` ;
d) `nested_24` : la moyenne de `cross_val_score(GridSearchCV(le même pipeline, la même grille, cv=FOLDS_24), X_24, Y_24, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=1))`, une validation croisée **imbriquée**.

La vérification contrôle a) à d), puis trace les deux courbes selon $k$.

Dans tes notes : pourquoi a) est-il si haut ? Pourquoi c) dépasse-t-il un peu b), et pourquoi d) ne le fait-il pas ? Lequel de ces quatre nombres annoncerais-tu dans un rapport ? Compare avec ce que tu avais trouvé en 🔬 8.25.

In [ ]:
rng_24 = np.random.default_rng(1524)
X_24 = rng_24.normal(size=(200, 2000))                     # 200 samples, 2,000 features of pure noise
Y_24 = rng_24.permutation(np.repeat([0, 1], 100))          # labels drawn at random: there is nothing to learn
FOLDS_24 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
KS_24 = [5, 10, 20, 50, 100, 200]

In [ ]:
leaky_24 = ...     # a) SelectKBest(f_classif, k=20) fitted on ALL the rows of X_24, then the mean of
                   #    cross_val_score(LogisticRegression(max_iter=1000), the 20 selected columns, Y_24, cv=FOLDS_24)
honest_24 = ...    # b) the mean of cross_val_score of make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
                   #    on X_24, Y_24, cv=FOLDS_24
search_24 = ...    # c) GridSearchCV of that pipeline over k in KS_24 (selectkbest__k), cv=FOLDS_24, fitted on X_24, Y_24
inner_24 = ...     #    its best_score_
nested_24 = ...    # d) the mean of cross_val_score(GridSearchCV(the same pipeline and grid, cv=FOLDS_24), X_24, Y_24,
                   #    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=1))

In [ ]:
wb.check("15.24a", leaky_24)
wb.check("15.24b", honest_24)
wb.check("15.24c", inner_24)
wb.check("15.24d", nested_24)
with wb.attempt("15.24"):
    if not filled(leaky_24, honest_24):
        raise NotImplementedError("leaky_24, honest_24")
    curve_24 = []
    for k in KS_24:
        chosen_24 = SelectKBest(f_classif, k=k).fit_transform(X_24, Y_24)
        curve_24.append((cross_val_score(LogisticRegression(max_iter=1000), chosen_24, Y_24, cv=FOLDS_24).mean(),
                         cross_val_score(make_pipeline(SelectKBest(f_classif, k=k), LogisticRegression(max_iter=1000)),
                                         X_24, Y_24, cv=FOLDS_24).mean()))
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    ax.plot(KS_24, [a for a, _ in curve_24], "o-", label="selection on all the rows, then cross-validation")
    ax.plot(KS_24, [b for _, b in curve_24], "s-", label="selection inside the pipeline")
    ax.axhline(0.5, color="gray", ls="--", label="chance level")
    ax.set(xscale="log", xlabel="number k of selected features", ylabel="cross-validated accuracy", ylim=(0, 1.05),
           title="200 samples, 2,000 noise features, random labels")
    ax.legend(fontsize=8)
    plt.show()

## Partie D · Décider, préparer, brancher, sauvegarder

Le seuil de décision (15.25) et les classes déséquilibrées (15.26), six familles de modèles sur les mêmes folds (15.27), le `ColumnTransformer` sur les manchots bruts (15.28), un estimateur de ta librairie dans scikit-learn (15.29), la sauvegarde d'un pipeline (15.30), puis le défi (15.31). La cellule suivante importe les derniers outils.

In [ ]:
import joblib
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import make_scorer
from sklearn.model_selection import FixedThresholdClassifier, RepeatedStratifiedKFold, TunedThresholdClassifierCV, cross_val_predict
from sklearn.model_selection._search import BaseSearchCV
from sklearn.utils.class_weight import compute_class_weight

### Ex 15.25 — Régler le seuil de décision d'un classifieur 🔬 ★★★ ⏱️ 35 min
**Objectif :** choisir le seuil d'un classifieur d'après le coût de ses erreurs, à la main puis avec `TunedThresholdClassifierCV`.  
**Prérequis :** Ex 15.10 · ch. 3 (matrice de confusion, recall, seuil) · fiche, section « seuil de décision » · **Fil rouge :** synthétique · **Parcours :** R, C

Un problème binaire avec environ 15 % de positifs. Manquer un positif (un faux négatif) coûte **10**, une fausse alerte (un faux positif) coûte **1** ; `cost_25(y_true, y_pred)` calcule le coût total. Les 3 000 exemples sont coupés en trois : 1 470 pour entraîner (`X_FIT_25`), 630 pour valider (`X_VAL_25`), 900 pour le test, qui ne sert qu'à mesurer. `MODEL_25` est une régression logistique déjà entraînée sur la première partie. **Contrat** :
a) `default_cost_25` : le coût de `MODEL_25.predict` sur le test (le seuil 0,5) ;
b) `val_costs_25` : pour chaque seuil $t$ de `THRESHOLDS_25` (0,01 à 0,99), le coût sur la **validation** de la règle « classe 1 quand `predict_proba(X)[:, 1] >= t` » ;
c) `best_threshold_25` : le seuil de plus petit coût de validation (le premier, en cas d'égalité) ;
d) `test_cost_25` : le coût sur le test de la règle avec ce seuil ;
e) `theory_threshold_25` : le seuil qui minimise le coût **moyen** si les probabilités sont calibrées : compare l'espérance du coût des deux décisions pour un exemple de probabilité $p$ (un nombre exact) ;
f) `recalls_25` : la liste `[recall au seuil 0,5, recall au seuil de c)]` sur le test ;
`tuned_25` : `TunedThresholdClassifierCV(make_pipeline(StandardScaler(), LogisticRegression()), scoring=COST_SCORER_25, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0))`, entraîné sur l'entraînement et la validation réunis (`X_REST_25`, `Y_REST_25`) ; g) `tuned_threshold_25` : son `best_threshold_` ; h) `tuned_cost_25` : le coût de `tuned_25.predict` sur le test.

La vérification contrôle a) à h), puis trace le coût de validation selon le seuil.

Dans tes notes : pourquoi le meilleur seuil est-il si loin de 0,5 ? Compare c) et e) : qu'est-ce qui les rapproche, et qu'est-ce qui les sépare ? Qu'a fait `TunedThresholdClassifierCV` de plus que toi, et avec quelles données ? Que coûte un meilleur recall ?

In [ ]:
X_25, Y_25 = make_classification(n_samples=3000, n_features=8, n_informative=4, weights=[0.85], class_sep=0.8,
                              flip_y=0.02, random_state=1525)      # about 15 % of positives (class 1)
X_REST_25, X_TEST_25, Y_REST_25, Y_TEST_25 = train_test_split(X_25, Y_25, test_size=0.3, random_state=1525, stratify=Y_25)
X_FIT_25, X_VAL_25, Y_FIT_25, Y_VAL_25 = train_test_split(X_REST_25, Y_REST_25, test_size=0.3, random_state=1525,
                                                          stratify=Y_REST_25)
COST_FN_25, COST_FP_25 = 10, 1        # a missed positive costs 10, a false alarm costs 1
THRESHOLDS_25 = np.round(np.arange(0.01, 1.0, 0.01), 2)


def cost_25(y_true, y_pred):
    """Total cost of the decisions y_pred: 10 per false negative, 1 per false positive."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return int(COST_FN_25 * fn + COST_FP_25 * fp)


COST_SCORER_25 = make_scorer(lambda y_true, y_pred: -cost_25(y_true, y_pred))   # a scorer: larger is better
MODEL_25 = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_FIT_25, Y_FIT_25)
print(len(Y_FIT_25), "to fit,", len(Y_VAL_25), "to validate,", len(Y_TEST_25), "to test")

In [ ]:
default_cost_25 = ...      # a) the cost on the test of MODEL_25.predict (the threshold 0.5)
val_costs_25 = ...         # b) list: for each threshold t of THRESHOLDS_25, the cost on the VALIDATION part of the rule
                           #    "class 1 when MODEL_25.predict_proba(X)[:, 1] >= t"
best_threshold_25 = ...    # c) the threshold of THRESHOLDS_25 with the smallest validation cost (the first one if tied)
test_cost_25 = ...         # d) the cost on the test of the rule with best_threshold_25
theory_threshold_25 = ...  # e) the threshold that minimises the expected cost when the probabilities are calibrated
recalls_25 = ...           # f) list [recall at 0.5, recall at best_threshold_25], on the test
tuned_25 = ...             #    TunedThresholdClassifierCV(make_pipeline(StandardScaler(), LogisticRegression()),
                           #    scoring=COST_SCORER_25, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0)),
                           #    fitted on X_REST_25, Y_REST_25 (the fitting and validation parts together)
tuned_threshold_25 = ...   # g) its best_threshold_
tuned_cost_25 = ...        # h) the cost of tuned_25.predict on the test

In [ ]:
wb.check("15.25a", default_cost_25)
wb.check("15.25b", val_costs_25)
wb.check("15.25c", best_threshold_25)
wb.check("15.25d", test_cost_25)
wb.check("15.25e", theory_threshold_25)
wb.check("15.25f", recalls_25)
wb.check("15.25g", tuned_threshold_25)
wb.check("15.25h", tuned_cost_25)
with wb.attempt("15.25"):
    if not filled(val_costs_25):
        raise NotImplementedError("val_costs_25")
    if len(val_costs_25) != len(THRESHOLDS_25):
        print(f"❌ Ex 15.25 : val_costs_25 doit avoir un coût par seuil de THRESHOLDS_25 ({len(THRESHOLDS_25)}).")
    else:
        fig, ax = plt.subplots(figsize=(6.5, 3.6))
        ax.plot(THRESHOLDS_25, val_costs_25, label="validation cost")
        ax.axvline(0.5, color="gray", ls="--", label="threshold of predict (0.5)")
        if filled(theory_threshold_25):
            ax.axvline(float(theory_threshold_25), color="tab:green", ls=":", label="your theoretical threshold")
        ax.set(xlabel="threshold on predict_proba", ylabel="cost (10 per FN, 1 per FP)")
        ax.legend(fontsize=8)
        plt.show()

### Ex 15.26 — Classes déséquilibrées : class_weight, rééchantillonnage et la bonne métrique (recall, PR-AUC) 📦 ★★★ ⏱️ 35 min
**Objectif :** comparer trois façons de traiter une classe rare, et les juger avec des mesures qui ne se laissent pas tromper.  
**Prérequis :** Ex 15.25 · ch. 3 (precision, recall, courbe precision-recall) · fiche, section « classes déséquilibrées » · **Fil rouge :** synthétique · **Parcours :** R, C

10 000 exemples, dont 2 % de positifs (la classe 1), coupés en 7 000 pour l'entraînement et 3 000 pour le test, en gardant les proportions. `report_26(modèle, X, y)` entraîne le modèle sur `(X, y)`, puis rend cinq mesures sur le test : `[accuracy, recall, precision, balanced accuracy, average precision]` (la dernière, l'aire sous la courbe precision-recall du ch. 3, se calcule sur `predict_proba`). **Contrat** (chaque réponse est la liste rendue par `report_26`) :
a) `dummy_26` : `DummyClassifier(strategy="most_frequent")`, entraîné sur l'entraînement ;
b) `plain_26` : `make_pipeline(StandardScaler(), LogisticRegression())` ;
c) `balanced_26` : le même pipeline, avec `LogisticRegression(class_weight="balanced")` ;
d) `under_26` : le pipeline de b), entraîné sur un entraînement **sous-échantillonné** : tous les positifs, plus autant de négatifs tirés sans remise avec `np.random.default_rng(1526).choice(indices des négatifs, size=nombre de positifs, replace=False)` (le test ne change pas) ;
e) `moved_26` : `FixedThresholdClassifier(le pipeline de b), threshold=0.1)`, entraîné sur tout l'entraînement ;
f) `weights_26` : la liste des poids que `class_weight="balanced"` donne aux classes 0 et 1 sur l'entraînement (`compute_class_weight` les calcule ; vérifie la formule de la fiche).

La vérification contrôle a) à f), puis affiche le tableau des cinq modèles.

Dans tes notes : que vaut l'accuracy de a), et que dit-elle ? Qu'ont en commun c), d) et e) ? Pourquoi l'average precision bouge-t-elle peu, alors que le recall et la precision changent beaucoup ? Quelle méthode choisirais-tu, et d'après quel coût ?

In [ ]:
X_26, Y_26 = make_classification(n_samples=10_000, n_features=10, n_informative=4, n_redundant=2, weights=[0.98],
                              class_sep=2.0, flip_y=0.0, random_state=1526)     # 2 % of positives (class 1)
X_TRAIN_26, X_TEST_26, Y_TRAIN_26, Y_TEST_26 = train_test_split(X_26, Y_26, test_size=0.3, random_state=1526, stratify=Y_26)
print(int(Y_TRAIN_26.sum()), "positives out of", len(Y_TRAIN_26), "training examples;", int(Y_TEST_26.sum()), "out of",
      len(Y_TEST_26), "in the test")


def report_26(model, X_fit, y_fit):
    """Fit `model` on (X_fit, y_fit), then [accuracy, recall, precision, balanced accuracy, average precision] on the test
    (the average precision on predict_proba[:, 1])."""
    model.fit(X_fit, y_fit)
    predicted, scores = model.predict(X_TEST_26), model.predict_proba(X_TEST_26)[:, 1]
    return [accuracy_score(Y_TEST_26, predicted), recall_score(Y_TEST_26, predicted, zero_division=0),
            precision_score(Y_TEST_26, predicted, zero_division=0), balanced_accuracy_score(Y_TEST_26, predicted),
            average_precision_score(Y_TEST_26, scores)]

In [ ]:
dummy_26 = ...     # a) report_26 of DummyClassifier(strategy="most_frequent"), trained on the training part
plain_26 = ...     # b) report_26 of make_pipeline(StandardScaler(), LogisticRegression())
balanced_26 = ...  # c) report_26 of the same pipeline with LogisticRegression(class_weight="balanced")
under_26 = ...     # d) report_26 of the pipeline of b), trained on an undersampled training part: all the positives, plus as
                   #    many negatives drawn with np.random.default_rng(1526).choice(negative indices, size=..., replace=False)
moved_26 = ...     # e) report_26 of FixedThresholdClassifier(the pipeline of b), threshold=0.1)
weights_26 = ...   # f) list: the weights that class_weight="balanced" gives to the classes 0 and 1 on the training part

In [ ]:
for letter_26, name_26 in zip("abcde", ["dummy_26", "plain_26", "balanced_26", "under_26", "moved_26"]):
    wb.check(f"15.26{letter_26}", globals()[name_26])
wb.check("15.26f", weights_26)
with wb.attempt("15.26"):
    if not filled(plain_26, balanced_26):
        raise NotImplementedError("plain_26, balanced_26")
    rows_26 = {name: globals()[name + "_26"] for name in ["dummy", "plain", "balanced", "under", "moved"]
               if filled(globals()[name + "_26"])}
    print(pd.DataFrame(rows_26, index=["accuracy", "recall", "precision", "balanced acc.", "average precision"]).T.round(3))

### Ex 15.27 — La même validation croisée pour six familles de modèles scikit-learn : score moyen, écart-type et temps 📦 ★★★ ⏱️ 40 min
**Objectif :** comparer honnêtement des familles de modèles, sur les mêmes folds, en score, en dispersion et en temps.  
**Prérequis :** ch. 12 (mise à l'échelle) · ch. 13 (classifieurs) · ch. 14 (ensembles) · Ex 15.9 · fiche §15.1, §15.6 à §15.8 · **Fil rouge :** Penguins · **Parcours :** C

Prédire le **sexe** des 333 manchots d'après leurs 4 mesures, sans l'espèce : un problème plus difficile que l'espèce. Six familles de modèles, chacune avec ses réglages par défaut (`FAMILIES_27`), et les mêmes 15 folds pour toutes (`FOLDS_27`, 5 folds répétés 3 fois). **Contrat** :
`compare_27(models, X, y, cv)` : pour chaque nom et chaque modèle du dictionnaire, `cross_validate(model, X, y, cv=cv)` ; rend un `DataFrame` indexé par les noms, aux colonnes `"mean"` et `"std"` (`ddof=1`) des accuracies de validation et `"fit_ms"` (la durée moyenne de `fit`, en millisecondes), trié par `"mean"` décroissant ; la vérification en tire a) les moyennes et b) les écarts-types, dans l'ordre de `FAMILIES_27` ;
c) `raw_knn_27` : la moyenne des accuracies de validation de `KNeighborsClassifier()` **sans** mise à l'échelle, sur `FOLDS_27` ;
d) `pca_svm_27` : la même pour `make_pipeline(StandardScaler(), PCA(n_components=2), SVC())`.

La vérification appelle `compare_27` sur les six familles, contrôle le tableau (forme, ordre, moyennes et écarts-types ; les durées dépendent de la machine et ne sont pas vérifiées), puis c) et d).

Dans tes notes : les écarts entre les quatre premières familles sont-ils plus grands que leurs écarts-types ? Quelle famille se détache des autres, et pourquoi (pense à ce que mélangent les mesures quand on ne connaît pas l'espèce) ? Combien de fois plus lent est l'ensemble le plus lent que la régression logistique, pour quel gain ? Que t'apprennent c) et d) sur la préparation ?

In [ ]:
X_27 = PENGUINS[MEASURES].to_numpy()          # the 4 measures of the 333 penguins
Y_27 = PENGUINS["sex"].to_numpy()             # the target: male or female
FOLDS_27 = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=1527)   # the same 15 folds for all
FAMILIES_27 = {"logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
               "knn": make_pipeline(StandardScaler(), KNeighborsClassifier()),
               "svm": make_pipeline(StandardScaler(), SVC()),
               "naive_bayes": GaussianNB(),
               "forest": RandomForestClassifier(random_state=0),
               "boosting": HistGradientBoostingClassifier(random_state=0)}

In [ ]:
def compare_27(models, X, y, cv):
    """For each name and model of the dict `models`, cross_validate(model, X, y, cv=cv). Return a pandas DataFrame indexed
    by the names, with the columns "mean" and "std" (ddof=1) of the validation accuracies and "fit_ms" (the mean fit time,
    in milliseconds), sorted by decreasing "mean"."""
    raise NotImplementedError("compare_27")


raw_knn_27 = ...   # c) the mean validation accuracy of KNeighborsClassifier() WITHOUT any scaling, on the folds FOLDS_27
pca_svm_27 = ...   # d) the same for make_pipeline(StandardScaler(), PCA(n_components=2), SVC())

In [ ]:
with checked("15.27"):
    table_27 = compare_27(FAMILIES_27, X_27, Y_27, FOLDS_27)
    if returned("15.27", "compare_27", table_27):
        shape_ok_27 = (isinstance(table_27, pd.DataFrame) and list(table_27.columns) == ["mean", "std", "fit_ms"]
                       and sorted(table_27.index) == sorted(FAMILIES_27))
        verdict("15.27", shape_ok_27, "compare_27 rend une ligne par famille et les colonnes mean, std, fit_ms.",
                f"compare_27 doit rendre un DataFrame indexé par les noms de FAMILIES_27, aux colonnes mean, std, fit_ms.")
        if shape_ok_27:
            verdict("15.27", list(table_27["mean"]) == sorted(table_27["mean"], reverse=True),
                    "le tableau est trié par score moyen décroissant.", "trie le tableau par score moyen décroissant.")
            print(table_27.round({"mean": 3, "std": 3, "fit_ms": 1}))
            wb.check("15.27a", [float(table_27.loc[name, "mean"]) for name in FAMILIES_27], computed=True)
            wb.check("15.27b", [float(table_27.loc[name, "std"]) for name in FAMILIES_27], computed=True)
wb.check("15.27c", raw_knn_27)
wb.check("15.27d", pca_svm_27)

### Ex 15.28 — ColumnTransformer sur Penguins brut 📦 ★★★ ⏱️ 45 min
**Objectif :** préparer des données hétérogènes, avec des valeurs manquantes et des catégories, dans un seul pipeline sans fuite.  
**Prérequis :** Ex 15.22 · ch. 12 (imputation, one-hot, fichier brut des manchots) · fiche, section « données hétérogènes » · **Fil rouge :** Penguins · **Parcours :** R, C

Le fichier brut des manchots du ch. 12 (`penguins_raw`, 344 lignes), sans ses colonnes d'identifiants, de dates et de commentaires, ni ses colonnes constantes : 6 colonnes de nombres (avec des valeurs manquantes) et 3 de texte (`Island`, `Clutch Completion`, `Sex`, avec des valeurs manquantes). La cible : l'espèce. **Contrat** :
`numeric_28` : un pipeline `SimpleImputer(strategy="median")` puis `StandardScaler()` ; `categorical_28` : un pipeline `SimpleImputer(strategy="most_frequent")` puis `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` ; `prep_28` : un `ColumnTransformer` à deux branches nommées `"num"` et `"cat"`, les colonnes choisies par `make_column_selector` (les nombres pour `"num"`, le texte, `dtype_include=object`, pour `"cat"`) ;
`model_28` : `make_pipeline(prep_28, LogisticRegression(max_iter=1000))`, entraîné sur `X_RAW_28`, `Y_28` ; a) `n_columns_28` : le nombre de colonnes que produit le `ColumnTransformer` entraîné (`model_28[0]`) ; `names_28` : la liste de leurs noms (`get_feature_names_out`) ;
b) `cv_28` : la moyenne de `cross_val_score(clone(model_28), X_RAW_28, Y_28, cv=FOLDS_28)` ;
c) `new_28` : la liste des espèces que `model_28` prédit pour les deux manchots de `NEW_28`, dont les colonnes sont dans un autre ordre, avec une île jamais vue et des valeurs manquantes ;
`search_28` : une `GridSearchCV(clone(model_28), ..., cv=FOLDS_28)` sur la stratégie de l'imputeur des nombres (`["mean", "median"]`) et le `C` de la régression logistique (`[0.01, 1, 100]`), entraînée sur `X_RAW_28`, `Y_28` (les noms des hyperparamètres : `pipe.get_params().keys()`) ; d) `ties_28` : le nombre de candidats classés premiers ; e) `weak_28` : le score moyen de validation du candidat qui a la médiane et `C = 0.01`.

La vérification contrôle a) à e), et les noms de `names_28`.

Dans tes notes : d'où vient le nombre de colonnes de a) ? Que se passe-t-il pour l'île inconnue de `NEW_28` ? Pourquoi l'ordre des colonnes de `NEW_28` ne gêne-t-il pas ? Pourquoi cette validation croisée est-elle honnête, alors que l'imputation regarde des moyennes ou des médianes ?

In [ ]:
RAW_28 = wb.datasets.load_penguins_raw()                       # 344 penguins, the file of the ch. 12
DROP_28 = ["studyName", "Sample Number", "Region", "Stage", "Individual ID", "Date Egg", "Comments", "Species"]
X_RAW_28 = RAW_28.drop(columns=DROP_28)                         # 6 numeric columns and 3 text columns, with NaN
Y_28 = RAW_28["Species"].str.split().str[0].to_numpy()          # Adelie, Chinstrap or Gentoo
FOLDS_28 = StratifiedKFold(n_splits=5, shuffle=True, random_state=1528)
NEW_28 = pd.DataFrame({"Sex": [np.nan, "FEMALE"], "Island": ["Anvers", "Torgersen"],    # two new penguins, the columns
                       "Flipper Length (mm)": [216.0, 186.0], "Body Mass (g)": [5300.0, 3450.0],   # in another order,
                       "Culmen Depth (mm)": [15.2, 18.1], "Culmen Length (mm)": [47.5, 38.6],      # an island never
                       "Clutch Completion": ["Yes", np.nan], "Delta 15 N (o/oo)": [np.nan, 8.9],   # seen, NaN
                       "Delta 13 C (o/oo)": [np.nan, -25.8]})
print(X_RAW_28.shape, "· missing values per column:", X_RAW_28.isna().sum().to_dict())

In [ ]:
numeric_28 = ...      #    a pipeline: SimpleImputer(strategy="median"), then StandardScaler()
categorical_28 = ...  #    a pipeline: SimpleImputer(strategy="most_frequent"), then
                      #    OneHotEncoder(handle_unknown="ignore", sparse_output=False)
prep_28 = ...         #    ColumnTransformer([("num", numeric_28, ...), ("cat", categorical_28, ...)]), the columns chosen
                      #    by make_column_selector (numbers for "num", text, dtype_include=object, for "cat")
model_28 = ...        #    make_pipeline(prep_28, LogisticRegression(max_iter=1000)), fitted on X_RAW_28, Y_28
n_columns_28 = ...    # a) the number of columns that the fitted ColumnTransformer (model_28[0]) produces
names_28 = ...        #    the list of their names (get_feature_names_out)
cv_28 = ...           # b) the mean of cross_val_score of clone(model_28) on X_RAW_28, Y_28, cv=FOLDS_28
new_28 = ...          # c) the list of the species that model_28 predicts for the two penguins of NEW_28
search_28 = ...       #    GridSearchCV(clone(model_28), a grid of the strategy of the numeric imputer (["mean", "median"])
                      #    and of C of the logistic regression ([0.01, 1, 100]), cv=FOLDS_28), fitted on X_RAW_28, Y_28
ties_28 = ...         # d) the number of candidates ranked first (rank_test_score equal to 1)
weak_28 = ...         # e) the mean validation score of the candidate with the median and C = 0.01

In [ ]:
wb.check("15.28a", n_columns_28)
with wb.attempt("15.28"):
    if not filled(names_28):
        raise NotImplementedError("names_28")
    names_shown_28 = [str(name) for name in names_28]
    expected_28 = {"num__Culmen Length (mm)", "num__Delta 13 C (o/oo)", "cat__Island_Biscoe", "cat__Sex_MALE",
                   "cat__Clutch Completion_No"}
    verdict("15.28", expected_28 <= set(names_shown_28),
            "les noms de colonnes sont ceux du ColumnTransformer (nom de la branche, deux tirets bas, colonne).",
            "names_28 doit venir de get_feature_names_out() du ColumnTransformer entraîné, branches nommées \"num\" et \"cat\".")
wb.check("15.28b", cv_28)
if filled(new_28) and (isinstance(new_28, str) or np.ndim(new_28) != 1 or len(new_28) != 2):
    print("❌ Ex 15.28c : new_28 doit être une liste de deux espèces, une par manchot de NEW_28, dans l'ordre des lignes.")
else:
    wb.check("15.28c", ", ".join(str(species) for species in new_28) if isinstance(new_28, (list, tuple, np.ndarray, pd.Series)) else new_28)
wb.check("15.28d", ties_28)
wb.check("15.28e", weak_28)

### Ex 15.29 — Quand scikit-learn refuse un estimateur mylearn 🐛 ★★★ ⏱️ 35 min
**Objectif :** comprendre ce que scikit-learn attend d'un estimateur, et réparer un adaptateur qui ne le respecte pas.  
**Prérequis :** ch. 13 (ton `KNeighborsClassifier`, 13.14) · Ex 15.17 · 0A (héritage) · fiche, section « brancher ton propre estimateur » · **Fil rouge :** Penguins · **Parcours :** C

Ta librairie `mylearn` n'hérite de rien de scikit-learn : ses classes suivent les conventions, mais n'ont ni `get_params` ni `set_params`. Les données : les 333 manchots **dans l'ordre du fichier** (triés par espèce), décrits par les deux mesures du bec, en millimètres tous les deux (pas besoin de mise à l'échelle). **Contrat** :
a) `error_29` : le **nom** de l'exception (une chaîne, comme `"ValueError"`) que lève `cross_val_score(mylearn.neighbors.KNeighborsClassifier(), X_29, Y_29, cv=5)` ; essaie-le dans une cellule à toi, et lis le message ;
ensuite, un collègue a écrit l'adaptateur `MylearnKNN_29`, qui enveloppe ton `KNeighborsClassifier` du ch. 13 pour le donner à scikit-learn. Il a **trois** bugs : trouve-les et corrige-les, sans changer son nom de classe ni ce qu'il calcule ; son hyperparamètre doit s'appeler `n_neighbors`, comme dans scikit-learn.

La vérification contrôle a), puis trois points, dans l'ordre où ils se masquent : 1) `clone` et `get_params` marchent ; 2) scikit-learn reconnaît l'adaptateur comme un classifieur, et `cross_val_score(..., cv=5)` donne les scores de folds stratifiés (comparés à `by_hand_29`, qui entraîne ton `KNeighborsClassifier` à la main dans chaque fold) ; 3) une recherche sur `n_neighbors` donne bien des scores différents, les mêmes qu'à la main. Le point 1 doit passer pour que les deux autres soient vérifiés ; quand les trois passent, elle relève b) la moyenne de `cross_val_score(MylearnKNN_29(), X_29, Y_29, cv=5)` et c) le `n_neighbors` choisi par la recherche. Elle attend (⏳) que ton `KNeighborsClassifier` soit écrit (13.14).

Dans tes notes : pour chaque bug, le symptôme, et la règle de la fiche qu'il viole. Lesquels font une erreur, lesquels se taisent ? Pourquoi le bug du point 2 donne-t-il des scores si mauvais sur ces données ?

In [ ]:
X_29 = PENGUINS[["bill_length_mm", "bill_depth_mm"]].to_numpy()   # two measures in millimetres: no scaling needed
Y_29 = PENGUINS["species"].to_numpy()                                # sorted by species, as in the file
KS_29 = [1, 5, 25, 75]


def by_hand_29(k, n_splits=5):
    """The oracle: StratifiedKFold(n_splits) without shuffling, YOUR mylearn KNeighborsClassifier fitted by hand in each
    fold; returns the n_splits accuracies."""
    scores = []
    for train, val in StratifiedKFold(n_splits=n_splits).split(X_29, Y_29):
        knn = mylearn.neighbors.KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_29[train], Y_29[train])
        scores.append(float(np.mean(np.asarray(knn.predict(X_29[val])) == Y_29[val])))
    return scores

In [ ]:
error_29 = ...   # a) the NAME of the exception (a string, e.g. "ValueError") that
                 #    cross_val_score(mylearn.neighbors.KNeighborsClassifier(), X_29, Y_29, cv=5) raises: try it in a cell
                 #    of your own


# A colleague's adapter, to give your KNeighborsClassifier to scikit-learn. It has three bugs: find and fix them
# (keep the class name; its hyperparameter must be called n_neighbors, as in scikit-learn).
class MylearnKNN_29(BaseEstimator, ClassifierMixin):
    """scikit-learn adapter around mylearn.neighbors.KNeighborsClassifier (the k-nearest neighbours of ch. 13)."""

    def __init__(self, k=5):
        self.n_neighbors = k
        self.model = mylearn.neighbors.KNeighborsClassifier(n_neighbors=k)

    def fit(self, X, y):
        self.model.fit(X, y)
        self.classes_ = self.model.classes_
        return self

    def predict(self, X):
        return self.model.predict(X)

In [ ]:
wb.check("15.29a", error_29)
with checked("15.29"):
    mylearn.neighbors.KNeighborsClassifier(n_neighbors=1).fit(X_29[:10], Y_29[:10]).predict(X_29[:2])   # ⏳ until 13.14 is written
    POINTS_29 = ["clone et get_params marchent : le constructeur range son argument sous son nom",
                 "scikit-learn reconnaît l'adaptateur comme un classifieur (folds stratifiés)",
                 "set_params change vraiment le modèle : une recherche sur n_neighbors donne des scores différents"]
    try:
        clone(MylearnKNN_29())                           # what cross_val_score does first, with the default arguments
        params_29 = clone(MylearnKNN_29(n_neighbors=7)).get_params()
        point1_29 = params_29 == {"n_neighbors": 7}
        why1_29 = f"get_params rend {params_29} au lieu de {{'n_neighbors': 7}}"
    except Exception as exc:
        point1_29, why1_29 = False, f"{type(exc).__name__} : {str(exc)[:200]}"
    verdict("15.29", point1_29, f"point 1 : {POINTS_29[0]}.", f"point 1 : {why1_29}.")
    if not point1_29:
        print("⏳ Ex 15.29 : points 2 et 3 : corrige d'abord le point 1 (scikit-learn ne peut même pas copier l'objet).")
    else:
        mine_29 = cross_val_score(MylearnKNN_29(n_neighbors=5), X_29, Y_29, cv=5)
        classifier_29 = is_classifier(MylearnKNN_29())
        why2_29 = (f"is_classifier(MylearnKNN_29()) vaut False : relis l'ordre d'héritage (fiche, « brancher ton propre estimateur »)"
                   if not classifier_29 else "vérifie que l'adaptateur calcule la même chose que ton KNeighborsClassifier")
        verdict("15.29", classifier_29 and np.allclose(mine_29, by_hand_29(5)),
                f"point 2 : {POINTS_29[1]}.",
                f"point 2 : avec cv=5, tes scores sont {np.round(mine_29, 3).tolist()} au lieu de "
                f"{np.round(by_hand_29(5), 3).tolist()} (folds stratifiés) ; {why2_29}.")
        search_29 = GridSearchCV(MylearnKNN_29(), {"n_neighbors": KS_29}, cv=StratifiedKFold(n_splits=5)).fit(X_29, Y_29)
        found_29 = search_29.cv_results_["mean_test_score"]
        expected_29 = [np.mean(by_hand_29(k)) for k in KS_29]
        stored_29 = sorted(vars(MylearnKNN_29(n_neighbors=3)))
        if not np.allclose(found_29, expected_29):
            why3_29 = (f"la recherche donne {np.round(found_29, 3).tolist()} pour n_neighbors = {KS_29}, au lieu de "
                       f"{np.round(expected_29, 3).tolist()} : le modèle utilise-t-il vraiment self.n_neighbors au moment de fit ?")
        else:
            why3_29 = (f"__init__ range {stored_29} : il ne doit rien faire d'autre que ranger son argument n_neighbors "
                       f"(le modèle se construit dans fit).")
        verdict("15.29", np.allclose(found_29, expected_29) and stored_29 == ["n_neighbors"], f"point 3 : {POINTS_29[2]}.",
                f"point 3 : {why3_29}")
        wb.check("15.29b", float(np.mean(mine_29)), computed=True)
        wb.check("15.29c", search_29.best_params_["n_neighbors"])

### Ex 15.30 — Sauvegarder, recharger et versionner un pipeline 🛠️ ★★ ⏱️ 20 min
**Objectif :** sauvegarder un pipeline entraîné avec ses métadonnées, le recharger, et être prévenu d'un changement de version.  
**Prérequis :** Ex 15.28 · fiche, section « sauvegarder un modèle et ses versions » · **Fil rouge :** Penguins · **Parcours :** R, C

`MODEL_30` est un pipeline entraîné comme celui de 15.28 (le prétraitement et la régression logistique). Écris deux fonctions, dont le contrat est dans leurs docstrings :
- `save_30(model, folder, cv_score)` écrit dans le dossier `folder` (un `pathlib.Path`) le pipeline **entier** (`model.joblib`, avec `joblib.dump`) et ses métadonnées (`model.json`) : les versions de Python, NumPy, scikit-learn et joblib, le score de validation croisée, la liste des colonnes d'entrée (`model.feature_names_in_`) et la date ;
- `load_30(folder)` relit les deux fichiers ; si la version de scikit-learn écrite dans `model.json` diffère de celle installée, elle affiche une ligne qui commence par `⚠️` et donne les deux versions ; elle rend le pipeline.

La vérification travaille dans un dossier temporaire (rien n'est écrit dans ton dépôt) : elle sauvegarde `MODEL_30`, contrôle les deux fichiers, recharge le pipeline et compare ses prédictions, puis écrit une fausse version dans `model.json` et vérifie l'avertissement. Cet exercice se corrige avec `05_solutions.md`.

Dans tes notes : pourquoi sauvegarder le pipeline entier plutôt que la seule régression logistique ? Que ferais-tu si l'avertissement apparaissait en production ? Pourquoi ne faut-il jamais charger un fichier `joblib` reçu d'un inconnu ? Que mettrais-tu dans le dépôt git, et que laisserais-tu dehors ?

In [ ]:
import datetime

MODEL_30 = make_pipeline(ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
                                             make_column_selector(dtype_include="number")),
                                            ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                                                                  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                                             make_column_selector(dtype_include=object))]),
                         LogisticRegression(max_iter=1000)).fit(X_RAW_28, Y_28)    # a fitted pipeline, as in 15.28
CV_SCORE_30 = 0.99   # its cross-validated accuracy, as you would have measured it (rounded)

In [ ]:
def save_30(model, folder, cv_score):
    """Write two files in `folder` (a pathlib.Path): model.joblib (joblib.dump of the WHOLE pipeline) and model.json, a dict
    with the keys "python", "numpy", "scikit-learn", "joblib" (the versions), "cv_accuracy" (cv_score), "features" (the
    list model.feature_names_in_) and "saved_on" (datetime.date.today().isoformat()). Return that dict."""
    raise NotImplementedError("save_30")


def load_30(folder):
    """Read model.json and model.joblib in `folder`. If the "scikit-learn" version written in model.json differs from the
    installed one (sklearn.__version__), print a line that starts with "⚠️" and gives both versions. Return the model."""
    raise NotImplementedError("load_30")

In [ ]:
import io
from pathlib import Path

with checked("15.30"):
    with tempfile.TemporaryDirectory() as tmp_30:
        folder_30 = Path(tmp_30)
        meta_30 = save_30(MODEL_30, folder_30, CV_SCORE_30)
        if returned("15.30", "save_30", meta_30):
            files_ok_30 = (folder_30 / "model.joblib").exists() and (folder_30 / "model.json").exists()
            verdict("15.30", files_ok_30, "save_30 écrit model.joblib et model.json.",
                    "save_30 doit écrire model.joblib et model.json dans le dossier reçu.")
            if files_ok_30:
                written_30 = json.loads((folder_30 / "model.json").read_text(encoding="utf-8"))
                keys_30 = ["python", "numpy", "scikit-learn", "joblib", "cv_accuracy", "features", "saved_on"]
                verdict("15.30", sorted(written_30) == sorted(keys_30) and written_30["scikit-learn"] == sklearn.__version__
                        and written_30["features"] == list(MODEL_30.feature_names_in_),
                        "model.json contient les versions, le score, les colonnes et la date.",
                        f"model.json doit contenir exactement les clés {keys_30}, avec les versions installées et les "
                        "colonnes d'entrée du pipeline.")
                again_30 = load_30(folder_30)
                if returned("15.30", "load_30", again_30):
                    verdict("15.30", isinstance(again_30, Pipeline) and np.array_equal(again_30.predict(X_RAW_28), MODEL_30.predict(X_RAW_28)),
                            "le pipeline rechargé prédit exactement comme l'original.",
                            "load_30 doit rendre le pipeline entier, qui prédit comme l'original.")
                    old_30 = dict(written_30, **{"scikit-learn": "0.19.2"})
                    (folder_30 / "model.json").write_text(json.dumps(old_30), encoding="utf-8")
                    shown_30 = io.StringIO()
                    with contextlib.redirect_stdout(shown_30):
                        load_30(folder_30)
                    said_30 = shown_30.getvalue()
                    verdict("15.30", "⚠️" in said_30 and "0.19.2" in said_30 and sklearn.__version__ in said_30,
                            "une version différente est signalée au chargement.",
                            "load_30 doit afficher une ligne ⚠️ avec les deux versions quand celle du fichier diffère.")

### Ex 15.31 — Défi : prédire le sexe des manchots depuis le fichier brut 🏆 ★★★★ ⏱️ 100 min
**Objectif :** construire une seule recherche sur un pipeline qui prend le fichier brut et donne des probabilités fiables, évaluée par validation croisée imbriquée.  
**Prérequis :** Ex 15.28 · Ex 15.24 (validation croisée imbriquée) · Ex 15.10 (probabilités) · ch. 12 (🏆 12.33, fuites du fichier brut) · **Fil rouge :** Penguins · **Parcours :** C

Au 🏆 12.33, le modèle était imposé et tu écrivais la préparation. Ici, tout est à toi, mais dans **une seule** pièce : `build_31()` rend une recherche d'hyperparamètres **neuve** (`GridSearchCV`, `RandomizedSearchCV` ou une recherche par successive halving) dont l'estimateur est un `Pipeline` qui contient un `ColumnTransformer`. Elle reçoit le fichier brut `X_31` (12 colonnes : texte, nombres, valeurs manquantes ; les colonnes qui trahissaient le sexe par la saisie, comme le numéro dans le nid, sont retirées et les lignes mélangées) et prédit `"MALE"` ou `"FEMALE"`. Chaque score est une **validation croisée imbriquée** : dans chaque fold extérieur, ta recherche entière est réentraînée sur les autres folds, puis notée sur celui-ci.

**Objectif**, sur le score caché : une accuracy d'au moins **0,92** **et** une log loss (moyenne de $-\ln \hat p(\text{vrai sexe})$, ch. 6) d'au plus **0,21** : des probabilités fiables, pas seulement des classes justes. Palier 🌟 : log loss au plus 0,195. Pour te situer : un `ColumnTransformer` simple (médiane et `StandardScaler` pour les nombres, one-hot pour le texte) suivi d'une régression logistique, dans une `GridSearchCV` qui règle `C` parmi 0,1, 1 et 10 avec `scoring="neg_log_loss"`, fait 0,918 et 0,220 sur le score caché.

Règles : la cellule affiche à chaque exécution le **score public** (5 folds × 2) ; règle ta méthode avec lui et avec ta propre validation croisée dans une cellule à toi. Le **score caché** (5 folds × 6, d'autres graines) n'est calculé qu'une fois par session, quand tu écris `REVEAL_31 = True` ; relancer la cellule redonne le même résultat, celui de ta méthode telle qu'elle était la première fois (redémarre le noyau pour une nouvelle tentative). La vérification refuse une recherche déjà entraînée, autre chose qu'une recherche sur un pipeline avec un `ColumnTransformer`, ou le même objet rendu deux fois, et réentraîne ta méthode sur des sexes mélangés au hasard (mélangés aussi dans `Y_31`, `DATA_31` et `RAW_31`, le temps de ce contrôle) : elle ne doit rien y apprendre ; le reste est une règle d'honneur : ne lis ni le score caché ni ses folds pour régler ta méthode. Compte une à deux minutes pour la vérification. Ce défi se corrige avec `05_solutions.md`.

Dans tes notes : les méthodes essayées et leurs scores publics ; ce qui a le plus aidé la log loss ; le temps de ta recherche.

In [ ]:
RAW_31 = wb.datasets.load_penguins_raw()
DATA_31 = RAW_31.drop(columns=["Sample Number", "Individual ID", "Comments", "Date Egg"])   # leaks of the collection (ch. 12)
DATA_31 = DATA_31[DATA_31["Sex"].isin(["MALE", "FEMALE"])].sample(frac=1, random_state=1531)
ROWS_31 = DATA_31.index                       # the rows of RAW_31, in the order of X_31 (used by the check)
DATA_31 = DATA_31.reset_index(drop=True)
X_31 = DATA_31.drop(columns="Sex")            # 333 penguins, 12 raw columns (text, numbers, NaN), rows shuffled
Y_31 = DATA_31["Sex"].to_numpy()              # "MALE" or "FEMALE"
PUBLIC_31 = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=31)
print(X_31.shape, "· columns:", list(X_31.columns))

In [ ]:
def build_31():
    """Return a NEW, unfitted hyperparameter search (GridSearchCV, RandomizedSearchCV or a Halving search) whose estimator is
    a Pipeline that contains a ColumnTransformer. It receives the raw DataFrame X_31 (its 12 columns, NaN included) and
    predicts "MALE" or "FEMALE"; its predict_proba is scored by the log loss."""
    raise NotImplementedError("build_31")

In [ ]:
from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted


def rules_31(search):
    """None when `search` follows the rules of the challenge, otherwise the reason (in French)."""
    if isinstance(search, type):
        return "build_31 doit rendre un objet (une instance), pas une classe"
    if not isinstance(search, BaseSearchCV):
        return f"build_31 rend un {type(search).__name__} : il faut une recherche d'hyperparamètres (GridSearchCV, RandomizedSearchCV…)"
    if not isinstance(search.estimator, Pipeline):
        return f"l'estimateur de la recherche est un {type(search.estimator).__name__} : il faut un Pipeline"
    if not any(isinstance(part, ColumnTransformer) for part in [*search.estimator.get_params(deep=True).values()]):
        return "le Pipeline ne contient pas de ColumnTransformer (ni directement, ni dans une de ses étapes)"
    try:
        check_is_fitted(search)
        return "build_31 rend une recherche déjà entraînée : elle doit être neuve"
    except NotFittedError:
        return None


def control_31():
    """Train build_31 on shuffled sexes, with Y_31, DATA_31 and RAW_31 shuffled the same way during the run (a method that
    reads them gets nothing true from them). Return the out-of-fold agreement with the shuffled sexes, then with the true
    ones: an honest method stays near 0.5 for both."""
    true_y = globals()["Y_31"]
    shuffled = np.random.default_rng(1531).permutation(true_y)
    saved = {name: globals()[name] for name in ("Y_31", "DATA_31", "RAW_31")}
    data, raw = saved["DATA_31"].copy(), saved["RAW_31"].copy()
    data["Sex"] = shuffled
    raw.loc[ROWS_31, "Sex"] = shuffled
    globals().update(Y_31=shuffled, DATA_31=data, RAW_31=raw)
    try:
        guessed = cross_val_predict(build_31(), X_31, shuffled, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=31))
    finally:
        globals().update(saved)
    return float(np.mean(guessed == shuffled)), float(np.mean(guessed == true_y))


def score_31(cv, y=Y_31):
    """Mean accuracy and mean log loss of a NEW search built by build_31, cross-validated on cv (the search is redone in
    every outer fold: a nested cross-validation)."""
    result = cross_validate(build_31(), X_31, y, cv=cv, scoring=["accuracy", "neg_log_loss"], error_score="raise")
    return float(result["test_accuracy"].mean()), float(-result["test_neg_log_loss"].mean())


with checked("15.31"):
    first_31, second_31 = build_31(), build_31()
    if returned("15.31", "build_31", first_31):
        broken_31 = rules_31(first_31) or ("build_31 rend deux fois le même objet : il doit en construire un neuf à chaque appel"
                                           if first_31 is second_31 else None)
        if broken_31:
            print(f"❌ Ex 15.31 : {broken_31}.")
        else:
            start_31 = time.perf_counter()
            public_31 = score_31(PUBLIC_31)
            print(f"public score (nested cross-validation, 5 folds x 2): accuracy {public_31[0]:.4f}, log loss "
                  f"{public_31[1]:.4f} ({time.perf_counter() - start_31:.0f} s)")
            if globals().get("REVEALED_31") is None and globals().get("REVEAL_31") is True:
                hidden_31 = score_31(RepeatedStratifiedKFold(n_splits=5, n_repeats=6, random_state=1531))
                REVEALED_31 = (hidden_31, control_31())          # once per session: kept if the cell runs again
            if globals().get("REVEALED_31") is not None:
                (accuracy_31, loss_31), (fake_31, real_31) = REVEALED_31
                print(f"hidden score (5 folds x 6, other seeds): accuracy {accuracy_31:.4f}, log loss {loss_31:.4f} "
                      f"(the method as it was the first time it was revealed in this session)")
                if fake_31 > 0.65 or abs(real_31 - 0.5) > 0.2:
                    print(f"⚠️ Ex 15.31 : le défi n'est pas validé : entraînée sur des sexes mélangés au hasard, ta méthode "
                          f"prédit encore ces sexes mélangés à {fr(fake_31)} et les vrais à {fr(real_31)} (une méthode honnête "
                          "reste vers 0,5 pour les deux) : lit-elle la cible ailleurs que dans le y reçu par fit, par exemple "
                          "dans Y_31, DATA_31 ou RAW_31 ?")
                else:
                    verdict("15.31", accuracy_31 >= 0.92 and loss_31 <= 0.21,
                            f"défi réussi : accuracy {fr(accuracy_31, 4)} et log loss {fr(loss_31, 4)} sur le score caché."
                            + (" 🌟 Palier atteint : log loss au plus 0,195." if loss_31 <= 0.195 else ""),
                            f"accuracy {fr(accuracy_31, 4)} (objectif ≥ 0,92) et log loss {fr(loss_31, 4)} (objectif ≤ "
                            "0,21) sur le score caché ; règle ta méthode sur le score public et ta propre validation "
                            "croisée, jamais sur le score caché (redémarre le noyau pour une nouvelle tentative).")
            elif globals().get("REVEAL_31") is not True:
                print("ℹ️ Ex 15.31 : quand ta méthode est prête, écris REVEAL_31 = True dans une cellule, puis relance celle-ci "
                      "pour le score caché (une seule fois par session).")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire ce que renvoient `fit`, `predict`, `decision_function`, `predict_proba` et `transform`, et sous quelle forme ?
2. Sais-tu construire un pipeline qui se règle par une recherche en grille ou au hasard sans fuite, et estimer le coût de cette recherche ?
3. Sais-tu moderniser un appel déprécié en lisant la documentation de ta version, et sauvegarder un modèle avec ses versions ?

**Pour aller plus loin** : les articles de Pedregosa et coll. (2011), de Buitinck et coll. (2013) et de Bergstra et Bengio (2012), la page « Common pitfalls » de la documentation, cités dans la fiche. La suite : le checkpoint de la partie III (ML classique), puis les réseaux de neurones (ch. 16).